# PointPillars training on livox dataset (Car only) — Training from scratch (randomized weights)

Trains PointPillars **from scratch with randomly initialized weights (no checkpoints, no pretrained models)** on the provided **livox** dataset with a **70/30 train/val split**, training **only for the "Car" class** (ignoring all others). Prints 3D / BEV / bbox mAP at **IoU >= 0.50 and IoU >= 0.70**, matched-detection counts at both thresholds, and BEV visualisations of the predictions on the validation split.

**Run top to bottom.** Edit only the *Config* cell: **paste the public Google Drive link**
of your dataset zip into `GDRIVE_ARCHIVE_URL` there, then run everything.

**No Drive storage?** Skip the link entirely: drag the zip into `/content` with Colab's
**Files panel** (folder icon, left sidebar) - the data cell picks it up automatically
(any file size), or use its upload button for smaller files (details in section 2).

**VS Code + Colab extension:** the kernel runs on the *Colab VM*, not on your PC, so
Windows paths are not visible to it. Get the data there with the Drive link (data cell).

What this version fixes vs the previous one:

* OpenPCDet's CUDA ops are actually **built** (the old notebook never ran `setup.py`,
  so `import pcdet` failed on the missing `pcdet/version.py`).
* `SharedArray` (a hard import of OpenPCDet) is installed. Without it `import pcdet.datasets` fails.
* Missing `ImageSets/{train,val}.txt` are generated from the infos (`CustomDataset` crashes
  on `sample_id_list[index]` when they are absent) - section 4a additionally creates the
  **70/30** split before the infos are built.
* `DontCare` annotations are stripped so the KITTI evaluator cannot score 0.
* `map_name_to_kitti` / numpy-string handling is patched (no `KeyError`).
* Removed numpy aliases (`np.int`, `np.float`, `np.bool`) are restored (needed on modern numpy).
* The training cell can no longer hang in OpenPCDet's post-training evaluation loop.
* Class / anchor / point-cloud-range config is derived from your data, and the KITTI class
  mapping used by the evaluator is valid (max 6 classes).
* A final cell prints a clean accuracy table (mAP + recall) at **IoU >= 0.50 and IoU >= 0.70**,
  section 12b counts matched detections at both thresholds, and section 13 draws them.
* The **simu CSV dataset** (`anno/<id>.txt` boxes + `points/<id>.txt` clouds) is recognised
  automatically and converted to the KITTI layout in section 4-conv - no manual export needed.
* **Faster alternative:** run `convert_simu_to_kitti.py` on your PC to build
  `simu_data_kitti.zip` (~10 GB) and upload that - section 4-conv is then skipped
  entirely (Drive link *or* Files-panel drag both work).


## 0. Environment


In [ ]:
import os, sys, re, json, glob, math, shutil, site, pickle, subprocess
from pathlib import Path
import numpy as np
import torch

print("python", sys.version.split()[0])
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU. In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU, then re-run this cell."
)
print(torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))

# The compiled OpenPCDet extensions link against torch's shared libraries (libc10.so,
# libtorch.so). Put torch/lib on the loader path so both this process and any subprocess
# can import them.
TORCH_LIB = str(Path(torch.__file__).parent / "lib")
if TORCH_LIB not in os.environ.get("LD_LIBRARY_PATH", "").split(":"):
    os.environ["LD_LIBRARY_PATH"] = (TORCH_LIB + ":" + os.environ.get("LD_LIBRARY_PATH", "")).rstrip(":")

# OpenPCDet checkpoints are pickled with numpy/EasyDict objects. torch>=2.6 defaults
# torch.load to weights_only=True and refuses them.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

# Many parts of OpenPCDet still use the numpy aliases np.int / np.float / np.bool.
# They were removed in numpy>=1.24 -> restore them.
for _n, _t in (("bool", bool), ("int", int), ("float", float),
               ("object", object), ("str", str), ("long", int)):
    if not hasattr(np, _n):
        setattr(np, _n, _t)

MAX_JOBS = str(max(1, min(4, os.cpu_count() or 2)))
os.environ.setdefault("MAX_JOBS", MAX_JOBS)


def sh(cmd, cwd=None, env=None, check=True, quiet=False):
    """Run a shell command, stream + capture its output, return the full text."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, errors="replace")
    buf = []
    for line in p.stdout:
        buf.append(line)
        if not quiet:
            print(line, end="")
    rc = p.wait()
    out = "".join(buf)
    if check and rc:
        raise RuntimeError("command failed (exit %d): %s\n----- last output -----\n%s"
                           % (rc, cmd, out[-4000:]))
    return out


def run_py(code_str, cwd=None, verbose=True):
    """Run a snippet in a fresh interpreter (same env), return (ok, output)."""
    r = subprocess.run([sys.executable, "-c", code_str], capture_output=True, text=True,
                       cwd=cwd, env=os.environ.copy())
    out = (r.stdout + r.stderr).strip()
    if verbose:
        print(out[-1500:])
    return r.returncode == 0, out

print("MAX_JOBS =", MAX_JOBS)

## 1. Config


In [ ]:
WORK = Path("/content") if Path("/content").is_dir() else Path.cwd()
DATA = WORK / "custom_livox_data"              # dataset location on the Colab VM
OPCD = WORK / "OpenPCDet"

for _p in (str(WORK), str(OPCD)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ==========================================================================
#  * PASTE THE PUBLIC GOOGLE DRIVE LINK OF YOUR DATASET ZIP HERE *
#    Drive -> Share -> Copy link
#    (e.g. https://drive.google.com/file/d/<ID>/view?usp=sharing - both
#     /file/d/.../view and open?id=... links work, they are fetched with gdown)
#    Leave the placeholder untouched if the zip already sits in MyDrive
#    (see DRIVE_ARCHIVE_CANDIDATES below as the fallback).
# ==========================================================================
GDRIVE_ARCHIVE_URL = "PASTE_YOUR_PUBLIC_DRIVE_LINK_HERE"

# --- fallbacks when no link is pasted: first file found in MyDrive is used ---
DRIVE_ARCHIVE_CANDIDATES = [
    "/content/drive/MyDrive/simu_data_kitti.zip",   # pre-converted KITTI zip (recommended)
    "/content/drive/MyDrive/livox_kitti.zip",
    "/content/drive/MyDrive/custom_livox_data.zip",
    "/content/drive/MyDrive/custom_livox_data.rar",
    "/content/drive/MyDrive/livox_data.zip",
    "/content/drive/MyDrive/livox_data.rar",
]

# --- 70 / 30 train-val split, created by section 4a ------------------------
SPLIT_TRAIN_FRAC = 0.7        # 70% of all frames -> train, 30% -> val
SPLIT_SEED = 42               # fixed seed = the identical split on every run
DUMMY_IMG_SIZE = (1242, 375)  # (W, H) placeholder size if image_2/*.png is missing

EPOCHS = 60                  # ~10k train frames: count on ~1-2 epochs per Colab hour
BATCH = 2                    # the ~1 km scene needs a 1600x1280 BEV grid; training runs
                             # in fp16 (--use_amp) and retries at half the batch on OOM
WORKERS = 2
USE_PRETRAINED = False        # NO checkpoint: random weights, trained from scratch
EXP_TAG = "scratch_livox_data"
KEEP_CLASSES = ["Car"]       # keep/train only these classes (None = every class in the data)

CKPT_SAVE_INTERVAL = 1       # 1 guarantees a checkpoint for the final epoch
MAX_CKPT_SAVE_NUM = 5
MAX_VOXELS_TRAIN = 60000     # wide scenes occupy many pillars even at 0.64 m
MAX_VOXELS_TEST = 100000

SPCONV_HINT = "auto"         # e.g. "spconv-cu124"; "auto" tries the matching CUDA build
INSTALL_SPCONV = True        # if no build works, a pure-numpy voxelizer is used automatically

# voxel size in metres; None = derived from the box extent in section 7.
# This dataset spans ~1 km, so the auto rule would pick 2.0 m pillars (far too coarse
# for cars) - 0.64 m gives a 1600x1280 BEV grid that fits a 16 GB GPU at BATCH = 2.
# Use 0.32 for finer pillars (then set BATCH = 1), or 1.0 for speed.
VOXEL_XY = 0.64

# Colab VMs have ~78 GB of disk and this dataset is huge (19 GB CSV zip / 95 GB of CSV,
# or the ~10 GB pre-converted KITTI zip -> 32 GB extracted):
# True drops the archive after staging/section 4-conv and the velodyne .bin copies after
# the npy files are written - saves ~50 GB. Set False only if your disk is large.
FREE_DISK_AFTER_CONVERT = True

print("WORK =", WORK)


## 2. Get the dataset onto the Colab VM

**Recommended: upload the pre-converted KITTI zip.** `convert_simu_to_kitti.py` (ships
next to these notebooks) converts `simu_data/` to the KITTI layout on your PC in a few
minutes and zips it to **`simu_data_kitti.zip` (~10 GB, the ~95 GB of point CSV compress
about 3x)**. Upload that zip and **section 4-conv is skipped entirely** - only the fast
sections 4a/4b run. Get it onto the VM either way:

* **Drive link:** upload to Drive -> Share -> *Anyone with the link* -> copy the link and
  paste it into `GDRIVE_ARCHIVE_URL` (Config cell), or
* **No Drive storage at all:** open Colab's **Files panel** (folder icon, left sidebar),
  drag `simu_data_kitti.zip` into `/content`, re-run this cell - picked up automatically,
  no link, no sign-in, works for any size.

**First paste your link:** open the *Config* cell (section 1) and replace
`PASTE_YOUR_PUBLIC_DRIVE_LINK_HERE` with the public share link of the zip you uploaded
to Google Drive (Drive -> Share -> *Copy link*). That is the only edit needed to run this
notebook from anywhere.

**No Drive storage? Drive is optional.** The data cell tries these in order:

1. the pasted link (gdown) - skipped if you leave the placeholder in place,
2. an already-mounted `MyDrive` (only if it is mounted - no forced sign-in),
3. **Colab's Files panel (folder icon, left sidebar): drag your zip into the
   `/content` folder and re-run the cell** - picked up automatically, works for any size
   (recommended for the 10 GB KITTI / 19 GB CSV zip),
4. an upload dialog in the data cell itself (browser upload - comfortable only up to a
   few GB; for anything bigger use option 3).

So the **zero-Drive workflow**: open the Files panel on the left, drag your zip into
`/content`, run the notebook top to bottom - no link, no Drive, no sign-in needed.

The archive may be either

* a **pre-converted KITTI** dataset from `convert_simu_to_kitti.py`
  (`simu_data_kitti.zip`, contains `training/{velodyne,label_2,calib,image_2}` + a
  ready-made `ImageSets/`) - **recommended**, section 4-conv is skipped, section 4a
  simply re-confirms the same 70/30 split (same seed),
* a **prepared** dataset (contains `custom_infos_{train,val}.pkl`), or
* a **simu CSV** dataset: a folder holding `anno/` (boxes per line:
  `id,class,x,y,z,l,w,h,yaw` - box **centres** in the LiDAR frame, classes
  `car / pedestrian / truck / bus / dog / bimo`) and `points/` (clouds per line:
  `x,y,z,intensity,ring,?`). Section **4-conv** converts it to the KITTI layout
  automatically; the point CSVs are streamed straight out of the zip, so the ~95 GB
  of point text is never unpacked on the VM disk - but it is **slow** (that is why the
  pre-converted zip above is recommended). **Zip it as .zip** (not .rar).
* a **raw KITTI-format** dataset: one folder holding
  `velodyne/ + label_2/ + calib/ + image_2/` (a `training/` wrapper folder around them is
  also fine, `ImageSets/` is optional - section **4a** creates the **70/30** split itself).
  The OpenPCDet infos are then generated automatically in section **4b**.
  `image_2` must hold one `.png` per frame; if images are missing the notebook writes
  placeholder images so the run still finishes (3D/BEV metrics stay valid, only bbox AP
  and the FOV point filter depend on the image size).

On your PC zip the folder with `tar` (older PowerShell writes backslash paths that break on Linux):

```
cd <parent folder>
tar -a -c -f custom_livox_data.zip <your_dataset_folder>
```

where `<your_dataset_folder>` directly contains `velodyne/`, `label_2/`, `calib/` and
`image_2/`. Upload it to Google Drive, share it by link and paste that link into
`GDRIVE_ARCHIVE_URL` (section 1) - `.zip` and `.rar` are both handled. **Without any
link** the notebook instead looks in an already-mounted `MyDrive`
(`custom_livox_data.zip` / `livox_data.zip`), then for the biggest `*.zip` sitting in
`/content` (drop it there with the Files panel), and finally opens an upload dialog.


In [ ]:
# Download the archive and stage it for OpenPCDet.
# Accepts a prepared dataset (custom_infos_*.pkl + custom_dbinfos_train.pkl + points/),
# a raw KITTI layout (<root>/{velodyne,label_2,calib,image_2}; training/ wrapper ok,
# ImageSets optional) or the simu CSV layout (anno/ + points/, converted to KITTI in
# section 4-conv). data_ready() checks ALL artifacts so a partial run is completed
# on re-run.
# RECOMMENDED: upload simu_data_kitti.zip (built on your PC with convert_simu_to_kitti.py)
# - it matches the raw KITTI layout above, so section 4-conv is skipped entirely and only
# the fast sections 4a/4b run. The archive is purged right after a raw layout is staged
# (FREE_DISK_AFTER_CONVERT) so sections 4b/6 keep the disk.

def data_ready(p):
    p = Path(p)
    return ((p / "custom_infos_train.pkl").exists()
            and (p / "custom_infos_val.pkl").exists()
            and (p / "custom_dbinfos_train.pkl").exists()
            and (p / "points").is_dir() and any((p / "points").glob("*.npy")))


def _first_tool(names):
    for n in names:
        if shutil.which(n):
            return n
    return None


def extract_archive(archive, dest):
    """Extract .zip or .rar without assuming a specific tool is installed."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    with open(archive, "rb") as f:
        magic = f.read(8)

    if magic[:2] == b"PK":                       # zip
        import zipfile
        with zipfile.ZipFile(archive) as z:
            z.extractall(dest)
        return "zipfile"

    if magic[:4] == b"Rar!":                     # rar
        tool = _first_tool(["unrar", "unar", "7z", "7zz", "7za", "bsdtar"])
        if tool is None and shutil.which("apt-get"):
            sh("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq unrar || true",
               check=False, quiet=True)
            tool = _first_tool(["unrar", "unar"])
        if tool is None:
            raise RuntimeError("Cannot extract RAR: none of unrar/unar/7z/bsdtar is available.")
        if tool == "unrar":
            sh('unrar x -o+ "%s" "%s/"' % (archive, dest), quiet=True)
        elif tool == "unar":
            sh('unar -f -o "%s" "%s"' % (dest, archive), quiet=True)
        else:
            sh('%s x -y -o"%s" "%s"' % (tool, dest, archive), quiet=True)
        return tool

    raise RuntimeError("Unsupported archive (magic=%r). Provide a .zip or .rar." % magic[:4])


def find_dir(base, required):
    base = Path(base)
    cands = [base] + sorted(q for q in base.rglob("*") if q.is_dir())
    for p in cands:
        if all((p / r).exists() for r in required):
            return p
    return None


def _archive_ok(path):
    """True if path really is a complete .zip/.rar - and NOT an HTML error page or a
    truncated download saved as one (the zip central directory check catches the latter)."""
    try:
        with open(path, "rb") as f:
            m = f.read(4)
        if m[:4] == b"Rar!":
            return True
        if m[:2] != b"PK":
            return False
        import zipfile
        with zipfile.ZipFile(path) as z:        # parses the central directory (fast)
            return len(z.namelist()) > 0
    except Exception:
        return False


def _html_reason(path):
    """Human-readable reason when Google served an HTML page instead of the file."""
    try:
        with open(path, "rb") as fb:
            magic = fb.read(2)
        with open(path, "r", errors="ignore") as f:
            txt = f.read(400000).lower()
    except OSError:
        return "unreadable file"
    if magic == b"PK":
        return ("truncated/partial download - valid zip start but the end of the file "
                "is missing (re-download it)")
    if not txt.lstrip().startswith("<"):
        return "binary file that is not a .zip/.rar (wrong file uploaded?)"
    if "too many users" in txt or "quota" in txt and "exceed" in txt:
        return ("Google Drive download QUOTA EXCEEDED - wait ~1 day, copy the zip to "
                "another Drive account, or skip Drive: drag the zip into /content with "
                "the Files panel")
    if ("you need access" in txt or "request access" in txt
            or "sign in" in txt and "accounts.google" in txt):
        return ("the link is NOT public: Drive -> Share -> General access -> "
                "'Anyone with the link' (Viewer), copy the link again and re-run")
    if "confirm=" in txt or "download-warning" in txt:
        return "Google's virus-scan confirmation page for a big file (retrying it below)"
    return "Google returned an HTML page instead of the file (link or permission problem)"


def _follow_drive_confirm(archive):
    """Large files sit behind a 'Download anyway' interstitial - follow that link."""
    import requests
    try:
        html = Path(archive).read_text(errors="ignore")
    except OSError:
        return False
    hrefs = (re.findall(r'href="([^"]+)"', html) + re.findall(r'action="([^"]+)"', html))
    hrefs = [h.replace("&amp;", "&") for h in hrefs]
    cands = [h for h in hrefs
             if "confirm=" in h or "export=download" in h or "usercontent.google.com" in h]
    if not cands:
        return False
    cands.sort(key=lambda h: ("drive.usercontent.google.com/download" in h,
                              "confirm=" in h, "export=download" in h), reverse=True)
    url = cands[0]
    if url.startswith("//"):
        url = "https:" + url
    elif url.startswith("/"):
        url = "https://drive.google.com" + url
    part = Path(str(archive) + ".part")
    try:
        with requests.Session() as s:
            with s.get(url, stream=True, timeout=(30, 300)) as r:
                r.raise_for_status()
                if "text/html" in r.headers.get("Content-Type", ""):
                    return False
                with open(part, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        if chunk:
                            f.write(chunk)
        if not _archive_ok(part) or part.stat().st_size <= 1024:
            part.unlink()
            return False
        part.replace(archive)
        return True
    except Exception as e:
        print("   confirm-link retry failed:", type(e).__name__, e)
        try:
            part.unlink()
        except OSError:
            pass
        return False


HAVE_RAW = False
archive = WORK / "ship_archive"


def _got_archive():
    return _archive_ok(archive) and archive.stat().st_size > 1024


CONVERTED = (DATA / ".simu_converted").exists()
CSV_STAGED = (DATA / "anno").is_dir() and any((DATA / "anno").glob("*.txt"))
RAW_STAGED = ((DATA / "training" / "velodyne").is_dir()
              and (DATA / "training" / "label_2").is_dir()
              and (DATA / "training" / "calib").is_dir()
              and any((DATA / "training" / "velodyne").glob("*.bin")))

if data_ready(DATA):
    print("prepared dataset already present:", DATA)
elif CONVERTED:
    HAVE_RAW = True
    print("simu CSV dataset already converted to the KITTI layout:", DATA)
elif RAW_STAGED and not CSV_STAGED:
    HAVE_RAW = True
    print("raw KITTI dataset already staged:", DATA, "(ImageSets is created in section 4a)")
else:
    # need the archive: reuse an earlier download, otherwise fetch it.
    # Order: (1) pasted Drive link via gdown, (2) an already-mounted MyDrive,
    # (3) a zip dropped into /content with Colab's Files panel (any size),
    # (4) the browser upload button (small files).
    # Validate a file left over from an earlier run: gdown sometimes saves Google's
    # HTML error/confirmation page as the "archive" - never feed that to the extractor.
    got = _got_archive()
    if archive.exists() and not got:
        print("discarding the previous ship_archive - it is not a real .zip/.rar:")
        print("   reason:", _html_reason(archive))
        archive.unlink()
    if got:
        print("reusing the archive from a previous run:", archive)
    else:
        have_link = bool(GDRIVE_ARCHIVE_URL) and GDRIVE_ARCHIVE_URL.strip().lower().startswith("http")

        if have_link:
            sh("%s -m pip install -q gdown" % sys.executable)
            for attempt in (1, 2):
                reason = ""
                try:
                    sh('gdown --fuzzy "%s" -O "%s"' % (GDRIVE_ARCHIVE_URL.strip(), archive), quiet=True)
                except Exception as e:
                    print("gdown attempt %d raised:" % attempt, type(e).__name__, e)
                if _got_archive():
                    got = True
                    print("gdown download ok: %.1f GB" % (archive.stat().st_size / 1e9))
                    break
                if archive.exists():
                    reason = _html_reason(archive)
                    print("gdown attempt %d got HTML instead of the zip: %s" % (attempt, reason))
                    if "confirmation" in reason and _follow_drive_confirm(archive):
                        got = True
                        print("downloaded through Google's confirm link: %.1f GB"
                              % (archive.stat().st_size / 1e9))
                        break
                    archive.unlink()          # keep nothing but real archives around
                if reason and ("NOT public" in reason or "QUOTA" in reason):
                    print("   -> not retrying; use one of the Drive-free options below instead")
                    break
        else:
            print("No Drive link pasted - Drive is NOT required: see the local options in "
                  "section 2 (Files panel drag & drop, or the upload button below).")

        # (2) MyDrive candidates - checked only when Drive is already mounted, so a
        #     missing/full Drive never forces a sign-in popup
        if not got and Path("/content/drive/MyDrive").exists():
            for cand in DRIVE_ARCHIVE_CANDIDATES:
                if Path(cand).exists():
                    shutil.copy(cand, archive)
                    got = _got_archive()
                    print("copied from Drive:", cand)
                    if not got:
                        print("   but it is not a real archive either -> skipped")
                        archive.unlink()
                    break
        # a pasted link implies you use Drive -> mount once and look for the candidates
        if not got and have_link:
            try:
                from google.colab import drive
                if not Path("/content/drive/MyDrive").exists():
                    drive.mount("/content/drive")
                for cand in DRIVE_ARCHIVE_CANDIDATES:
                    if Path(cand).exists():
                        shutil.copy(cand, archive)
                        got = _got_archive()
                        print("copied from Drive:", cand)
                        if not got:
                            archive.unlink()
                        break
            except Exception as e:
                print("Drive mount unavailable:", type(e).__name__, e)

        # (3) a zip already sitting in /content - drag it in with Colab's Files panel
        #     (folder icon, left sidebar); works for the full 19 GB file
        if not got:
            local = [p for p in set(WORK.glob("*.zip")) | set(WORK.glob("*/*.zip"))
                     if p.is_file() and p.resolve() != archive.resolve()]
            local.sort(key=lambda p: -p.stat().st_size)
            for p in local:
                if _archive_ok(p) and p.stat().st_size > 1024:
                    print("found a zip in /content: %s (%.1f GB) -> using it"
                          % (p, p.stat().st_size / 1e9))
                    shutil.move(str(p), str(archive))   # rename = instant, any size
                    got = _got_archive()
                    break
                print("ignoring %s - not a real .zip/.rar" % p)

        # (4) last resort: upload straight from this computer (browser dialog)
        if not got:
            print("Uploading from this computer - pick the zip in the dialog below. "
                  "(For the big 19 GB file prefer the Files panel on the left: drag it "
                  "into /content, cancel this dialog and re-run this cell.)")
            try:
                from google.colab import files
                uploaded = files.upload()
                if uploaded:
                    name = next(iter(uploaded))
                    if Path(name).exists():
                        shutil.move(name, str(archive))
                    else:
                        with open(str(archive), "wb") as uf:
                            uf.write(uploaded[name])
                    got = _got_archive()
                    if got:
                        print("uploaded: %s (%.1f GB)" % (name, archive.stat().st_size / 1e9))
                    elif archive.exists():
                        print("the uploaded file is not a real .zip/.rar -> ignored")
                        archive.unlink()
            except Exception as e:
                print("Upload unavailable:", type(e).__name__, e)

    assert got, (
        "Could not obtain the dataset archive. None of these is Drive-dependent:\n"
        "  1) paste a public link into GDRIVE_ARCHIVE_URL (Config cell), or\n"
        "  2) drag the zip into /content with Colab's Files panel (left sidebar) - "
        "recommended for the 19 GB file, or\n"
        "  3) run this cell again and choose the file in the upload dialog.\n"
        "Then re-run this cell.")

    # ---- which layout does the archive hold? (detect a zip by magic bytes) ---
    csv_zip = False
    try:
        import zipfile
        with open(archive, "rb") as f:
            is_zip = f.read(2) == b"PK"
        if is_zip:
            with zipfile.ZipFile(archive) as z:
                names = z.namelist()
            csv_zip = (not any(re.search(r"(^|/)velodyne/", n) for n in names)
                       and any(re.search(r"(^|/)points/", n) for n in names))
    except Exception as _e:
        print("layout detection failed (%s) - assuming raw KITTI layout" % _e)

    tmp = WORK / "ship_extract"
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True, exist_ok=True)

    if csv_zip:
        # simu CSV dataset: stage only the small anno/ folder - the points/ CSVs stay
        # inside the archive and are streamed by the converter (section 4-conv), so the
        # ~95 GB of point text never sits unpacked on the VM disk.
        print("layout: simu CSV dataset (anno/*.txt + points/*.txt)")
        import zipfile
        with zipfile.ZipFile(archive) as z:
            anno_entries = [n for n in z.namelist() if re.search(r"(^|/)anno/[^/]+\.txt$", n)]
            assert anno_entries, "no anno/*.txt entries found inside the archive"
            for n in anno_entries:
                with z.open(n) as s_f, open(tmp / Path(n).name, "wb") as d_f:
                    shutil.copyfileobj(s_f, d_f)
        print("extracted %d anno files - the point clouds are converted in section 4-conv"
              % len(anno_entries))
        shutil.rmtree(DATA, ignore_errors=True)
        DATA.mkdir(parents=True, exist_ok=True)
        shutil.move(str(tmp), str(DATA / "anno"))
    else:
        print("extracting with:", extract_archive(archive, tmp))

        prepared = find_dir(tmp, ("custom_infos_train.pkl",))
        if prepared is not None:
            shutil.rmtree(DATA, ignore_errors=True)
            shutil.move(str(prepared), str(DATA))
            print("found prepared dataset")
        else:
            raw = find_dir(tmp, ("velodyne", "label_2"))
            if raw is None:
                listing = [str(p.relative_to(tmp)) for p in sorted(tmp.rglob("*"))[:60]]
                raise FileNotFoundError(
                    "Neither a prepared dataset (custom_infos_train.pkl), a raw KITTI layout "
                    "(velodyne/ + label_2/) nor the simu CSV layout (anno/ + points/ - zip it "
                    "as .zip) was found in the archive. Contents: %s" % listing)
            shutil.rmtree(DATA, ignore_errors=True)
            (DATA / "training").mkdir(parents=True, exist_ok=True)
            # MOVE, never copy: a ~32 GB KITTI tree would otherwise sit on disk twice
            # and blow past the ~78 GB Colab disk before section 4b even starts.
            src_sets = raw / "ImageSets"
            if not src_sets.is_dir() and raw != tmp:
                src_sets = raw.parent / "ImageSets"   # standard KITTI: ImageSets sits next to training/
            if src_sets.is_dir():
                shutil.move(str(src_sets), str(DATA / "ImageSets"))
            for sub in ("velodyne", "label_2", "calib"):
                assert (raw / sub).is_dir(), "raw dataset is missing '%s/'" % sub
                shutil.move(str(raw / sub), str(DATA / "training" / sub))
            img_src = raw / "image_2" if (raw / "image_2").is_dir() else raw / "Image_2"
            if img_src.is_dir():
                shutil.move(str(img_src), str(DATA / "training" / "image_2"))
            HAVE_RAW = True
            print("staged raw KITTI dataset from:", raw)
            # For a raw layout the archive is not needed any more (nothing streams from
            # it the way section 4-conv does), so drop it to free disk for sections 4b/6.
            if FREE_DISK_AFTER_CONVERT and archive.exists():
                sz = archive.stat().st_size / 1e9
                archive.unlink()
                print("purged the archive to free %.1f GB - the data is fully staged"
                      " (it re-downloads automatically if staging ever has to run again)" % sz)
    shutil.rmtree(tmp, ignore_errors=True)

if data_ready(DATA):
    print("dataset ok:", DATA)
else:
    print("dataset staged at:", DATA)
    for p in sorted(DATA.iterdir()):
        print("  ", p.name)


## 3. Install OpenPCDet (builds the CUDA ops, ~5-10 min the first time)


In [ ]:
if not OPCD.exists():
    sh("git clone --depth 1 https://github.com/open-mmlab/OpenPCDet.git", cwd=str(WORK))

# SharedArray is a hard import of OpenPCDet (database_sampler.py) -> must be installed.
# Only install what is actually missing, so nothing in the Colab image gets downgraded.
def ensure(import_code, pip_name):
    if run_py(import_code, verbose=False)[0]:
        return True
    print("installing %s ..." % pip_name)
    sh("%s -m pip install -q %s" % (sys.executable, pip_name), check=False, quiet=True)
    ok = run_py(import_code, verbose=False)[0]
    print("  %s: %s" % (pip_name, "ok" if ok else "STILL MISSING"))
    return ok

assert ensure("import SharedArray", "SharedArray"), "SharedArray is required by OpenPCDet"
for _imp, _pip in [
    ("import numba", "numba"),
    ("import tensorboardX", "tensorboardX"),
    ("import easydict", "easydict"),
    ("import yaml", "pyyaml"),
    ("import skimage", "scikit-image"),
    ("import scipy", "scipy"),
    ("import tqdm", "tqdm"),
    ("import pyquaternion", "pyquaternion"),
    ("import matplotlib", "matplotlib"),
    ("import pandas", "pandas"),
    ("import ninja", "ninja"),
]:
    ensure(_imp, _pip)

# make `import pcdet` work in every process spawned from this kernel
try:
    site_dir = Path(site.getsitepackages()[0])
except Exception:
    site_dir = Path(sys.prefix) / "lib" / ("python%d.%d" % sys.version_info[:2]) / "site-packages"
try:
    site_dir.mkdir(parents=True, exist_ok=True)
    (site_dir / "openpcdet.pth").write_text(str(OPCD) + "\n")
except Exception as e:
    print("could not write openpcdet.pth (%s) - relying on PYTHONPATH instead" % type(e).__name__)
if str(OPCD) not in sys.path:
    sys.path.insert(0, str(OPCD))

# pcdet/__init__.py imports pcdet/version.py, which only setup.py creates
ver_file = OPCD / "pcdet" / "version.py"
if not ver_file.exists():
    ver_file.write_text('__version__ = "0.6.0"\n')

cuda_home = os.environ.get("CUDA_HOME")
if not cuda_home and Path("/usr/local/cuda/bin/nvcc").exists():
    cuda_home = "/usr/local/cuda"
print("CUDA_HOME:", cuda_home, "| nvcc:", shutil.which("nvcc") or "(not on PATH)")
build_env = {"MAX_JOBS": MAX_JOBS}
if cuda_home:
    build_env["CUDA_HOME"] = cuda_home

OPS_PROBE = (
    "import torch;"
    "import sys; sys.path.insert(0, r'%s');"
    "import pcdet;"
    "import pcdet.ops.iou3d_nms.iou3d_nms_cuda;"
    "import pcdet.ops.roiaware_pool3d.roiaware_pool3d_cuda;"
    "import pcdet.ops.pointnet2.pointnet2_stack.pointnet2_stack_cuda;"
    "import pcdet.ops.bev_pool.bev_pool_ext;"
    "print('OPS_OK')" % str(OPCD)
)

print("building CUDA extensions ...")
sh("%s setup.py build_ext --inplace" % sys.executable, cwd=str(OPCD), env=build_env, check=False)

ops_ok, _ = run_py(OPS_PROBE, cwd=str(OPCD))
if not ops_ok:
    print("build_ext was not enough, trying an editable pip install ...")
    sh("%s -m pip install -q -e . --no-build-isolation" % sys.executable, cwd=str(OPCD), check=False)
    ops_ok, _ = run_py(OPS_PROBE, cwd=str(OPCD))

assert ops_ok, (
    "OpenPCDet CUDA extensions are not importable. The build log is above; the usual causes are "
    "a missing CUDA toolkit (nvcc) or an nvcc/torch version mismatch."
)
print("OpenPCDet ready:", sh("git -C %s rev-parse --short HEAD" % OPCD, quiet=True).strip())

## 3b. Patch OpenPCDet imports (datasets, models, KITTI class names)

Runs right after the build so that later cells can import `pcdet` even though the optional
datasets (nuScenes, Waymo, Lyft, Argo2, ...) are not installed here.

* `pcdet/datasets/__init__.py` unconditionally imports nuscenes/waymo/lyft/argo2/... which need
  datasets not installed here -> the optional ones are made optional.
* `kitti_utils.transform_annotations_to_kitti_format` does `map[name]` on numpy string arrays;
  this is replaced with a safe, fixed-width-agnostic mapping.

* `anchor_generator.py` builds the anchor grid with `torch.arange(min, max + 1e-5, step)`.
  `point_cloud_range` is **float32**: for `|max| >= 256` the `+1e-5` guard vanishes
  (`ulp(512.0) = 6.1e-5`), the boundary point equals `stop` and is dropped -> one anchor
  row/column fewer than the BEV feature map -> the first training step dies in the focal
  loss with `The size of tensor a (1021122) must match ... (1024000)`. The shifts are now
  generated as exactly `feature_map_size` points (same math, deterministic count).


In [ ]:
DATASETS_INIT = """
import torch
from functools import partial
from torch.utils.data import DataLoader
from torch.utils.data import DistributedSampler as _DistributedSampler

from pcdet.utils import common_utils

from .dataset import DatasetTemplate
from .kitti.kitti_dataset import KittiDataset
from .custom.custom_dataset import CustomDataset

__all__ = {
    "DatasetTemplate": DatasetTemplate,
    "KittiDataset": KittiDataset,
    "CustomDataset": CustomDataset,
}

import importlib as _importlib

for _name, _mod, _cls in [
    ("NuScenesDataset", ".nuscenes.nuscenes_dataset", "NuScenesDataset"),
    ("WaymoDataset", ".waymo.waymo_dataset", "WaymoDataset"),
    ("PandasetDataset", ".pandaset.pandaset_dataset", "PandasetDataset"),
    ("LyftDataset", ".lyft.lyft_dataset", "LyftDataset"),
    ("ONCEDataset", ".once.once_dataset", "ONCEDataset"),
    ("Argo2Dataset", ".argo2.argo2_dataset", "Argo2Dataset"),
]:
    try:
        _m = _importlib.import_module(_mod, __package__)
        __all__[_name] = getattr(_m, _cls)
    except Exception as _e:
        print("optional dataset %s unavailable (%s)" % (_name, type(_e).__name__))


class DistributedSampler(_DistributedSampler):
    def __init__(self, dataset, num_replicas=None, rank=None, shuffle=True):
        super().__init__(dataset, num_replicas=num_replicas, rank=rank)
        self.shuffle = shuffle

    def __iter__(self):
        if self.shuffle:
            g = torch.Generator()
            g.manual_seed(self.epoch)
            indices = torch.randperm(len(self.dataset), generator=g).tolist()
        else:
            indices = torch.arange(len(self.dataset)).tolist()
        indices += indices[:(self.total_size - len(indices))]
        assert len(indices) == self.total_size
        indices = indices[self.rank:self.total_size:self.num_replicas]
        assert len(indices) == self.num_samples
        return iter(indices)


def build_dataloader(dataset_cfg, class_names, batch_size, dist, root_path=None, workers=4, seed=None,
                     logger=None, training=True, merge_all_iters_to_one_epoch=False, total_epochs=0):
    dataset = __all__[dataset_cfg.DATASET](
        dataset_cfg=dataset_cfg, class_names=class_names, root_path=root_path,
        training=training, logger=logger,
    )
    if merge_all_iters_to_one_epoch:
        assert hasattr(dataset, "merge_all_iters_to_one_epoch")
        dataset.merge_all_iters_to_one_epoch(merge=True, epochs=total_epochs)
    if dist:
        if training:
            sampler = torch.utils.data.distributed.DistributedSampler(dataset)
        else:
            rank, world_size = common_utils.get_dist_info()
            sampler = DistributedSampler(dataset, world_size, rank, shuffle=False)
    else:
        sampler = None
    dataloader = DataLoader(
        dataset, batch_size=batch_size, pin_memory=True, num_workers=workers,
        shuffle=(sampler is None) and training, collate_fn=dataset.collate_batch,
        drop_last=False, sampler=sampler, timeout=0,
        worker_init_fn=partial(common_utils.worker_init_fn, seed=seed),
    )
    return dataset, dataloader, sampler
"""
(OPCD / "pcdet" / "datasets" / "__init__.py").write_text(DATASETS_INIT)
print("patched pcdet/datasets/__init__.py")

ku = OPCD / "pcdet" / "datasets" / "kitti" / "kitti_utils.py"
ku_code = ku.read_text()
old_block = (
    "        for k in range(anno['name'].shape[0]):\n"
    "            anno['name'][k] = map_name_to_kitti[anno['name'][k]]\n"
)
new_block = (
    "        anno['name'] = np.array(\n"
    "            [map_name_to_kitti.get(str(n), str(n)) for n in anno['name']], dtype='<U32'\n"
    "        )\n"
)
if old_block in ku_code:
    ku.write_text(ku_code.replace(old_block, new_block))
    print("patched kitti_utils.py class-name mapping")
elif "map_name_to_kitti.get(str(n)" in ku_code:
    print("kitti_utils.py already patched")
else:
    print("WARNING: kitti_utils.py mapping block not found - patching skipped")


# The models package imports the whole detector/backbone zoo. Only the pieces needed by
# PointPillars are kept mandatory; everything else is imported best-effort so a single
# unavailable optional model can never break training.
def write_tolerant_init(path, required, optional, trailer=""):
    lines = ["import importlib as _importlib", "",
             "# generated by the notebook: keep OpenPCDet importable without optional models"]
    for mod, cls in required:
        lines.append("from .%s import %s" % (mod, cls))
    lines.append("")
    lines.append("__all__ = {")
    for mod, cls in required:
        lines.append("    %r: %s," % (cls, cls))
    lines.append("}")
    lines.append("")
    lines.append("for _mod, _cls in %r:" % (optional,))
    lines.append("    try:")
    lines.append("        _m = _importlib.import_module('.' + _mod, __package__)")
    lines.append("        __all__[_cls] = getattr(_m, _cls)")
    lines.append("    except Exception as _e:")
    lines.append("        print('optional module %s unavailable (%s)' % (_cls, type(_e).__name__))")
    if trailer:
        lines.append("")
        lines.append("")
        lines.append(trailer)
    path.write_text("\n".join(lines) + "\n")


BUILD_DETECTOR = """def build_detector(model_cfg, num_class, dataset):
    model = __all__[model_cfg.NAME](
        model_cfg=model_cfg, num_class=num_class, dataset=dataset
    )
    return model"""


M = OPCD / "pcdet" / "models"
write_tolerant_init(
    M / "detectors" / "__init__.py",
    [("detector3d_template", "Detector3DTemplate"), ("pointpillar", "PointPillar")],
    [("second_net", "SECONDNet"), ("PartA2_net", "PartA2Net"), ("point_rcnn", "PointRCNN"),
     ("pv_rcnn", "PVRCNN"), ("second_net_iou", "SECONDNetIoU"), ("caddn", "CaDDN"),
     ("voxel_rcnn", "VoxelRCNN"), ("centerpoint", "CenterPoint"),
     ("pv_rcnn_plusplus", "PVRCNNPlusPlus"), ("mppnet", "MPPNet"), ("mppnet_e2e", "MPPNetE2E"),
     ("pillarnet", "PillarNet"), ("voxelnext", "VoxelNeXt"), ("transfusion", "TransFusion"),
     ("bevfusion", "BevFusion")],
    trailer=BUILD_DETECTOR,
)
write_tolerant_init(
    M / "dense_heads" / "__init__.py",
    [("anchor_head_template", "AnchorHeadTemplate"), ("anchor_head_single", "AnchorHeadSingle")],
    [("anchor_head_multi", "AnchorHeadMulti"), ("point_head_box", "PointHeadBox"),
     ("point_head_simple", "PointHeadSimple"), ("point_intra_part_head", "PointIntraPartOffsetHead"),
     ("center_head", "CenterHead"), ("voxelnext_head", "VoxelNeXtHead"),
     ("transfusion_head", "TransFusionHead")],
)
write_tolerant_init(
    M / "backbones_3d" / "__init__.py", [],
    [("pointnet2_backbone", "PointNet2Backbone"), ("pointnet2_backbone", "PointNet2MSG"),
     ("spconv_backbone", "VoxelBackBone8x"), ("spconv_backbone", "VoxelResBackBone8x"),
     ("spconv_backbone_2d", "PillarBackBone8x"), ("spconv_backbone_2d", "PillarRes18BackBone8x"),
     ("spconv_backbone_focal", "VoxelBackBone8xFocal"),
     ("spconv_backbone_voxelnext", "VoxelResBackBone8xVoxelNeXt"),
     ("spconv_backbone_voxelnext2d", "VoxelResBackBone8xVoxelNeXt2D"),
     ("spconv_unet", "UNetV2"), ("dsvt", "DSVT")],
)
write_tolerant_init(
    M / "backbones_3d" / "vfe" / "__init__.py",
    [("vfe_template", "VFETemplate"), ("pillar_vfe", "PillarVFE")],
    [("mean_vfe", "MeanVFE"), ("dynamic_mean_vfe", "DynamicMeanVFE"),
     ("dynamic_pillar_vfe", "DynamicPillarVFE"), ("dynamic_pillar_vfe", "DynamicPillarVFESimple2D"),
     ("dynamic_voxel_vfe", "DynamicVoxelVFE"), ("image_vfe", "ImageVFE")],
)
write_tolerant_init(
    M / "backbones_3d" / "pfe" / "__init__.py", [],
    [("voxel_set_abstraction", "VoxelSetAbstraction")],
)
write_tolerant_init(
    M / "backbones_2d" / "map_to_bev" / "__init__.py",
    [("pointpillar_scatter", "PointPillarScatter")],
    [("height_compression", "HeightCompression"), ("pointpillar_scatter", "PointPillarScatter3d"),
     ("conv2d_collapse", "Conv2DCollapse")],
)
write_tolerant_init(
    M / "roi_heads" / "__init__.py", [],
    [("partA2_head", "PartA2FCHead"), ("pointrcnn_head", "PointRCNNHead"),
     ("pvrcnn_head", "PVRCNNHead"), ("second_head", "SECONDHead"),
     ("voxelrcnn_head", "VoxelRCNNHead"), ("roi_head_template", "RoIHeadTemplate"),
     ("mppnet_head", "MPPNetHead"), ("mppnet_memory_bank_e2e", "MPPNetHeadE2E")],
)
print("patched model subpackage __init__.py files for tolerant imports")

# --- fix: the anchor grid must be EXACTLY feature_map_size points per axis ---------------
AG = OPCD / "pcdet" / "models" / "dense_heads" / "target_assigner" / "anchor_generator.py"
_ag = AG.read_text()
if "1e-5" in _ag:
    # torch.arange(min, max + 1e-5, step): the +1e-5 guard dies in float32 when the
    # point-cloud range is large (ulp(512.0) = 6.1e-5 > 1e-5), the last anchor row/column
    # is excluded -> anchors 799x639 vs an 800x640 feature map -> focal-loss shape crash.
    _ag_x, _nx = re.subn(
        r"x_shifts = torch\.arange\(\s*self\.anchor_range\[0\] \+ x_offset,\s*"
        r"self\.anchor_range\[3\] \+ 1e-5, step=x_stride, dtype=torch\.float32,\s*\)\.cuda\(\)",
        "x_shifts = (torch.arange(int(grid_size[0]), dtype=torch.float32) * float(x_stride)\n"
        "            + float(self.anchor_range[0] + x_offset)).cuda()",
        _ag)
    _ag_y, _ny = re.subn(
        r"y_shifts = torch\.arange\(\s*self\.anchor_range\[1\] \+ y_offset,\s*"
        r"self\.anchor_range\[4\] \+ 1e-5, step=y_stride, dtype=torch\.float32,\s*\)\.cuda\(\)",
        "y_shifts = (torch.arange(int(grid_size[1]), dtype=torch.float32) * float(y_stride)\n"
        "            + float(self.anchor_range[1] + y_offset)).cuda()",
        _ag_x)
    if _nx == 1 and _ny == 1:
        AG.write_text(_ag_y)
        print("anchor_generator patched: anchor shifts are now exactly feature_map_size points")
    else:
        # upstream reformatted the file -> at least enlarge the guard so it survives float32
        _ag2 = _ag.replace("+ 1e-5", "+ 1e-3")
        assert "+ 1e-5" not in _ag2, "cannot patch %s - edit it by hand (see markdown above)" % AG
        AG.write_text(_ag2)
        print("anchor_generator patched (fallback): float32 guard 1e-5 -> 1e-3")
else:
    print("anchor_generator already patched (no 1e-5 guard found)")
assert "+ 1e-5" not in AG.read_text(), "anchor_generator still contains the 1e-5 guard: %s" % AG


## 4. spconv (optional — only used for voxelisation)

PointPillars does **not** use sparse convolutions. If a working `spconv` is available it is used,
otherwise a small numpy voxelizer with identical semantics is applied automatically.


In [ ]:
SPCONV_PROBE = (
    "import numpy as np, spconv.pytorch, spconv.utils, cumm.tensorview as tv;"
    "from spconv.utils import Point2VoxelCPU3d as V;"
    "v=V(vsize_xyz=[.16,.16,4],coors_range_xyz=[0,-39.68,-3,69.12,39.68,1],"
    "num_point_features=4,max_num_voxels=100,max_num_points_per_voxel=32);"
    "v.point_to_voxel(tv.from_numpy(np.random.rand(500,4).astype(np.float32)));"
    "print('SPCONV_OK')"
)

USE_SHIM = False
if not run_py(SPCONV_PROBE, cwd=str(WORK), verbose=False)[0]:
    if INSTALL_SPCONV:
        cu = (torch.version.cuda or "").replace(".", "")
        order = []
        if SPCONV_HINT != "auto":
            order.append(SPCONV_HINT)
        if len(cu) >= 3:
            order.append("spconv-cu" + cu[:3])
        order += ["spconv-cu124", "spconv-cu121", "spconv-cu120", "spconv-cu118"]
        for pkg in dict.fromkeys(order):
            print("trying %s ..." % pkg)
            sh("%s -m pip install -q %s" % (sys.executable, pkg), check=False, quiet=True)
            if run_py(SPCONV_PROBE, cwd=str(WORK), verbose=False)[0]:
                print("using", pkg)
                break
    USE_SHIM = not run_py(SPCONV_PROBE, cwd=str(WORK), verbose=False)[0]
print("USE_SHIM =", USE_SHIM)

SHIM_SRC = r"""
import sys
import types
from functools import partial

import numpy as np


def _stub_spconv():
    import torch.nn as nn

    class _Meta(type):
        def __getattr__(cls, attr):
            if attr.startswith("__"):
                raise AttributeError(attr)
            sub = _Meta(attr, (nn.Module,), {})
            setattr(cls, attr, sub)
            return sub

    def _module(name):
        mod = types.ModuleType(name)
        mod.__path__ = []
        mod.__version__ = "2.2.4"          # pcdet/utils/spconv_utils.py reads this

        def _getattr(attr):
            if attr.startswith("__"):
                raise AttributeError(attr)
            cls = _Meta(attr, (nn.Module,), {})
            setattr(mod, attr, cls)
            return cls

        mod.__getattr__ = _getattr
        return mod

    names = ["spconv", "spconv.pytorch", "spconv.pytorch.modules", "spconv.pytorch.utils",
             "spconv.utils", "spconv.core", "spconv.constants", "spconv.conv",
             "cumm", "cumm.tensorview"]
    for n in names:
        sys.modules[n] = _module(n)
    for n in names:
        parent, _, child = n.rpartition(".")
        if parent:
            setattr(sys.modules[parent], child, sys.modules[n])


def _voxelize(self, data_dict=None, config=None):
    # Same voxelisation semantics as spconv's Point2VoxelCPU3d, in pure numpy.
    pcr = np.asarray(self.point_cloud_range, dtype=np.float64)
    if data_dict is None:
        self.grid_size = np.round((pcr[3:6] - pcr[0:3]) / np.asarray(config.VOXEL_SIZE)).astype(np.int64)
        self.voxel_size = config.VOXEL_SIZE
        return partial(self.transform_points_to_voxels, config=config)

    pts = np.asarray(data_dict["points"], dtype=np.float32)
    vs = np.asarray(config.VOXEL_SIZE, dtype=np.float64)
    gs = np.asarray(self.grid_size, dtype=np.int64)
    max_pts = int(config.MAX_POINTS_PER_VOXEL)
    max_vox = int(config.MAX_NUMBER_OF_VOXELS[self.mode])

    ijk = np.floor((pts[:, :3].astype(np.float64) - pcr[:3]) / vs).astype(np.int64)
    keep = np.all((ijk >= 0) & (ijk < gs), axis=1)
    pts, ijk = pts[keep], ijk[keep]

    if pts.shape[0] == 0:
        data_dict["voxels"] = np.zeros((0, max_pts, pts.shape[1]), dtype=np.float32)
        data_dict["voxel_coords"] = np.zeros((0, 3), dtype=np.int32)
        data_dict["voxel_num_points"] = np.zeros((0,), dtype=np.int32)
        return data_dict

    key = ijk[:, 0] + ijk[:, 1] * gs[0] + ijk[:, 2] * gs[0] * gs[1]
    _, first, inv = np.unique(key, return_index=True, return_inverse=True)
    inv = inv.reshape(-1)
    order = np.argsort(first)                       # voxels in order of first appearance
    rank = np.empty(order.shape[0], dtype=np.int64)
    rank[order] = np.arange(order.shape[0])
    vid = rank[inv]

    nv = int(min(order.shape[0], max_vox))
    sel = np.nonzero(vid < nv)[0]
    sel = sel[np.argsort(vid[sel], kind="stable")]
    vsorted = vid[sel]

    starts = np.searchsorted(vsorted, np.arange(nv))
    within = np.arange(sel.shape[0]) - starts[vsorted]
    take = within < max_pts
    v, w, src = vsorted[take], within[take], sel[take]

    voxels = np.zeros((nv, max_pts, pts.shape[1]), dtype=np.float32)
    voxels[v, w] = pts[src]
    num = np.bincount(v, minlength=nv).astype(np.int32)
    coords = ijk[first[order][:nv]][:, ::-1].astype(np.int32)    # (z, y, x)

    if not data_dict.get("use_lead_xyz", True):
        voxels = voxels[..., 3:]

    data_dict["voxels"] = voxels
    data_dict["voxel_coords"] = coords
    data_dict["voxel_num_points"] = num
    return data_dict


def apply(use_shim):
    if use_shim:
        _stub_spconv()
        from pcdet.datasets.processor import data_processor as dp
        dp.DataProcessor.transform_points_to_voxels = _voxelize
"""

(WORK / "pcdet_shim.py").write_text(SHIM_SRC)

RUNNER_SRC = r"""
import os
import runpy
import sys
from pathlib import Path

import numpy as np
for _n, _t in (("bool", bool), ("int", int), ("float", float),
               ("object", object), ("str", str), ("long", int)):
    if not hasattr(np, _n):
        setattr(np, _n, _t)

os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

import torch
_torch_lib = str(Path(torch.__file__).parent / "lib")
if _torch_lib not in os.environ.get("LD_LIBRARY_PATH", ""):
    os.environ["LD_LIBRARY_PATH"] = (_torch_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")).rstrip(":")

_script = sys.argv[1]
for _p in [str(Path(_script).resolve().parent), "__REPO__/tools", "__REPO__", "__WORK__"]:
    if _p and _p not in sys.path:
        sys.path.insert(0, _p)

import pcdet_shim
pcdet_shim.apply(os.environ.get("PCDET_USE_SHIM", "0") == "1")

sys.argv = sys.argv[1:]
runpy.run_path(_script, run_name="__main__")
"""
RUNNER_SRC = RUNNER_SRC.replace("__REPO__", str(OPCD)).replace("__WORK__", str(WORK))
(WORK / "run_pcdet.py").write_text(RUNNER_SRC)
print("wrote", WORK / "pcdet_shim.py", "and", WORK / "run_pcdet.py")

## 4-conv. Convert the **simu CSV dataset** (`anno/` + `points/`) to KITTI format

> **Skip this cell entirely:** run `convert_simu_to_kitti.py` on your PC (next to these
> notebooks) and upload the resulting **`simu_data_kitti.zip`** instead - it already
> contains this exact `training/` layout, so this cell detects it and does nothing.

Only runs for CSV-layout datasets (raw KITTI zips skip this cell). Reads the
`anno/<id>.txt` boxes (`id,class,x,y,z,l,w,h,yaw` - box **centres** in the LiDAR frame)
and streams every `points/<id>.txt` cloud straight out of the downloaded archive,
writing a standard raw KITTI layout under `training/`:

* `velodyne/<id>.bin` - float32 `x, y, z, intensity` (the constant `ring` columns are dropped),
* `label_2/<id>.txt` - KITTI 15-field labels (`class trunc occ alpha x1 y1 x2 y2 h w l x y z ry`,
  with `z` = box **bottom** centre and `ry = -(yaw + pi/2)`, so the box comes out of
  OpenPCDet's KITTI-to-LiDAR transform exactly as it went in),
* `calib/<id>.txt` - synthetic identity calibration (your boxes are already LiDAR-frame).

The ~95 GB of point CSVs are **streamed** out of the zip and written as compact float32
`.bin` files (~32 GB); resume-safe, so an interrupted run continues where it stopped.


In [ ]:
# --- section 4-conv: simu CSV (anno/ + points/) -> raw KITTI layout --------------
import io, time, warnings, zipfile

SIMU_CLASS_MAP = {
    "car": "Car", "pedestrian": "Pedestrian", "cyclist": "Cyclist",
    "truck": "Truck", "bus": "Bus", "van": "Van", "dog": "Dog", "bimo": "Bimo",
}

# identity calibration: rect_to_lidar() becomes the identity, so a label written in the
# LiDAR frame passes through OpenPCDet's KITTI -> LiDAR transform unchanged
SIMU_CALIB_TEXT = "\n".join([
    "P0: 0 0 0 0 0 0 0 0 0 0 0 0",
    "P1: 0 0 0 0 0 0 0 0 0 0 0 0",
    "P2: 700 0 621 0 0 700 187.5 0 0 0 1 0",
    "P3: 0 0 0 0 0 0 0 0 0 0 0 0",
    "R0_rect: 1 0 0 0 1 0 0 0 1",
    "Tr_velo_to_cam: 1 0 0 0 0 1 0 0 0 0 1 0",
    "Tr_imu_to_velo: 1 0 0 0 0 1 0 0 0 0 1 0",
]) + "\n"


def simu_label_line(csv_line, W, H):
    """anno CSV line -> one KITTI 15-field label line (space separated)."""
    p = csv_line.strip().split(",")
    assert len(p) == 9, "expected 'id,class,x,y,z,l,w,h,yaw', got: %r" % csv_line
    cls = p[1].strip()
    name = SIMU_CLASS_MAP.get(cls.lower(), cls[:1].upper() + cls[1:])
    x, y, z, l, w, h, yaw = (float(v) for v in p[2:])
    ry = (-(yaw + math.pi / 2.0) + math.pi) % (2.0 * math.pi) - math.pi
    zb = z - h / 2.0                       # KITTI stores the box bottom centre
    alpha = -math.atan2(-y, x) + ry
    # synthetic 2D box from a plausible camera projection (f=700, 1242x375 image)
    depth = max(x, 0.5)
    fu, fv = 700.0 * (-y) / depth + 621.0, 700.0 * (-zb) / depth + 187.5
    bh, bw = max(700.0 * h / depth, 1.0), max(700.0 * w / depth, 1.0)
    x1 = min(max(fu - bw / 2.0, 0.0), W - 1.0); y1 = min(max(fv - bh / 2.0, 0.0), H - 1.0)
    x2 = min(max(fu + bw / 2.0, x1 + 1.0), W - 1.0); y2 = min(max(fv + bh / 2.0, y1 + 1.0), H - 1.0)
    return ("%s 0.00 0 %.4f %.2f %.2f %.2f %.2f %.3f %.3f %.3f %.3f %.3f %.3f %.4f"
            % (name, alpha, x1, y1, x2, y2, h, w, l, x, y, zb, ry))


def read_simu_points(raw):
    """'x,y,z,intensity,ring,?' CSV bytes -> (N, 4) float32 (x, y, z, intensity)."""
    n = raw.count(b"\n")
    clean = raw.strip().replace(b"\r", b"").replace(b"\n", b",")
    arr = None
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            arr = np.fromstring(clean, sep=",", dtype=np.float32)
    except Exception:
        arr = None
    if arr is None or arr.size != n * 6:           # fallback if the fast path fails
        arr = np.asarray(np.loadtxt(io.BytesIO(raw), delimiter=",", dtype=np.float32),
                         dtype=np.float32).reshape(-1)
    return arr.reshape(-1, 6)[:, :4]


def convert_simu_csv_to_kitti(anno_dir, archive, out_root):
    anno_dir, out_root = Path(anno_dir), Path(out_root)
    marker = out_root / ".simu_converted"
    if marker.exists():
        print("simu CSV -> KITTI conversion already finished:", marker)
        return
    velo = out_root / "training" / "velodyne"
    lab = out_root / "training" / "label_2"
    cal = out_root / "training" / "calib"
    for d in (velo, lab, cal):
        d.mkdir(parents=True, exist_ok=True)

    # --- every anno file that holds at least one box becomes a frame -----------
    keep_ids, anno_lines, dropped = [], {}, []
    for af in sorted(anno_dir.glob("*.txt")):
        lines = [l.strip() for l in af.read_text().splitlines() if l.strip()]
        if lines:
            keep_ids.append(af.stem)
            anno_lines[af.stem] = lines
        else:
            dropped.append(af.stem)
    if dropped:
        print("NOTE: dropped %d frame(s) without any box (e.g. %s)" % (len(dropped), dropped[:3]))
    assert keep_ids, "no usable anno/<id>.txt files in %s" % anno_dir

    seen = {}
    for sid in keep_ids:
        for l in anno_lines[sid]:
            c = l.split(",")[1].strip()
            seen[c] = seen.get(c, 0) + 1
    print("anno boxes by class:", dict(sorted(seen.items(), key=lambda kv: -kv[1])))

    W, H = DUMMY_IMG_SIZE
    for sid in keep_ids:
        lp = lab / (sid + ".txt")
        if not lp.exists():
            lp.write_text("\n".join(simu_label_line(l, W, H) for l in anno_lines[sid]) + "\n")
        cp = cal / (sid + ".txt")
        if not cp.exists():
            cp.write_text(SIMU_CALIB_TEXT)

    # --- stream the clouds out of the archive (never unpack the 95 GB of CSV) --
    with open(archive, "rb") as _f:
        assert _f.read(2) == b"PK", (
            "the CSV dataset must be zipped as .zip (got %s - re-zip it, .rar cannot "
            "be streamed)" % archive)
    with zipfile.ZipFile(archive) as zf:
        entries = {}
        for nm in zf.namelist():
            m = re.search(r"(^|/)points/([^/]+)\.txt$", nm)
            if m:
                entries[m.group(2)] = nm
        missing = [s for s in keep_ids if s not in entries]
        assert not missing, ("%d frame(s) have anno but no points entry (e.g. %s)"
                             % (len(missing), missing[:3]))
        t0 = time.time()
        for k, sid in enumerate(keep_ids):
            out = velo / (sid + ".bin")
            if out.exists():
                continue
            with zf.open(entries[sid]) as fh:
                read_simu_points(fh.read()).astype(np.float32).tofile(out)
            if (k + 1) % 250 == 0 or k + 1 == len(keep_ids):
                el = (time.time() - t0) / 60.0
                eta = el / (k + 1) * (len(keep_ids) - k - 1)
                print("  points %5d/%d  (%.1f min elapsed, ~%.0f min left)"
                      % (k + 1, len(keep_ids), el, eta))

    marker.write_text("converted %d frames from %s at %s"
                      % (len(keep_ids), archive.name, time.strftime("%Y-%m-%d %H:%M")))
    print("converted %d frames -> %s" % (len(keep_ids), out_root / "training"))
    if FREE_DISK_AFTER_CONVERT and archive.exists():
        sz = archive.stat().st_size / 1e9
        archive.unlink()
        print("purged the archive to free %.1f GB (it re-downloads automatically "
              "if the conversion ever needs to run again)" % sz)


if (DATA / "anno").is_dir() and not (DATA / ".simu_converted").exists():
    assert (WORK / "ship_archive").exists(), (
        "simu CSV data is staged but the archive is missing - re-run section 2 to fetch it again")
    convert_simu_csv_to_kitti(DATA / "anno", WORK / "ship_archive", DATA)

if (DATA / "training" / "velodyne").is_dir():
    HAVE_RAW = True
    n_bins = len(list((DATA / "training" / "velodyne").glob("*.bin")))
    print("frames ready for section 4a: %d" % n_bins)
    assert n_bins, "training/velodyne exists but is empty - re-run this cell"
else:
    print("no training/velodyne found - run section 2 (dataset download) first")


## 4a. Create the 70/30 train / val split (`ImageSets`)

Shuffles every frame id found in `training/velodyne` with the fixed seed from the Config
cell (`SPLIT_SEED`) and writes `ImageSets/{train,val,test}.txt`: **70% train / 30% val**.
Re-running the cell reproduces the identical split, so the zip does **not** need to contain
`ImageSets`. Also makes sure every frame has the `training/image_2/<id>.png` that
OpenPCDet's info generation (section 4b) requires - jpgs are converted, and if the dataset
ships no images at all placeholder images are written with a warning.


In [ ]:
# --- section 4a: 70/30 train-val split (+ make sure image_2 exists for section 4b) ---
import random

if data_ready(DATA):
    print("prepared dataset present - using its built-in train/val split as-is")
else:
    velo_dir = DATA / "training" / "velodyne"
    label_dir = DATA / "training" / "label_2"
    calib_dir = DATA / "training" / "calib"
    img_dir = DATA / "training" / "image_2"
    for d in (velo_dir, label_dir, calib_dir):
        assert d.is_dir(), ("missing %s - the zip must contain a raw KITTI layout: "
                            "velodyne/ label_2/ calib/ image_2/" % d)

    frame_ids = sorted(p.stem for p in velo_dir.glob("*.bin"))
    assert len(frame_ids) >= 2, "need at least 2 frames to split, found %d" % len(frame_ids)

    for d in (label_dir, calib_dir):
        have = {p.stem for p in d.glob("*.txt")}
        miss = [i for i in frame_ids if i not in have]
        assert not miss, ("%d frame(s) are missing %s/<id>.txt (e.g. %s)"
                          % (len(miss), d.name, miss[:3]))

    # KittiDataset.get_image_shape() hard-requires training/image_2/<id>.png (section 4b)
    png_ids = {p.stem for p in img_dir.glob("*.png")} if img_dir.is_dir() else set()
    missing = [i for i in frame_ids if i not in png_ids]
    if missing and img_dir.is_dir():
        jpg_map = {}
        for suf in (".jpg", ".jpeg"):
            for p in img_dir.glob("*" + suf):
                jpg_map.setdefault(p.stem, p)
        conv = [i for i in missing if i in jpg_map]
        if conv:
            from PIL import Image
            for i in conv:
                Image.open(jpg_map[i]).convert("RGB").save(img_dir / (i + ".png"))
            png_ids |= set(conv)
            missing = [i for i in missing if i not in png_ids]
            print("converted %d jpg image(s) to png" % len(conv))
    if missing:
        from PIL import Image
        img_dir.mkdir(parents=True, exist_ok=True)
        W, H = DUMMY_IMG_SIZE
        for i in missing:
            Image.new("RGB", (W, H), (0, 0, 0)).save(img_dir / (i + ".png"))
        print("WARNING: %d frame(s) had no image_2 png -> wrote placeholder %dx%d images."
              % (len(missing), W, H))
        print("         3D/BEV mAP is unaffected; bbox AP and the FOV point filter use this "
              "size - add the real images to the zip for exact numbers.")

    # --- 70/30 split with a fixed seed: identical on every run -----------------
    sets = DATA / "ImageSets"
    prev = {}
    for s in ("train", "val"):
        f = sets / (s + ".txt")
        prev[s] = len([ln for ln in f.read_text().splitlines() if ln.strip()]) if f.exists() else None
    if any(v is not None for v in prev.values()):
        print("existing ImageSets found (train=%s, val=%s) - overriding with a fresh %d/%d split"
              % (prev["train"], prev["val"],
                 round(100 * SPLIT_TRAIN_FRAC), round(100 * (1 - SPLIT_TRAIN_FRAC))))

    shuffled = frame_ids[:]
    random.Random(SPLIT_SEED).shuffle(shuffled)
    n_train = int(round(SPLIT_TRAIN_FRAC * len(shuffled)))
    n_train = min(max(n_train, 1), len(shuffled) - 1)   # keep >=1 train AND >=1 val frame
    train_ids, val_ids = shuffled[:n_train], shuffled[n_train:]

    sets.mkdir(exist_ok=True)
    (sets / "train.txt").write_text("\n".join(train_ids) + "\n")
    (sets / "val.txt").write_text("\n".join(val_ids) + "\n")
    (sets / "test.txt").write_text("\n".join(val_ids) + "\n")   # eval split == val

    print("70/30 split (seed %d): %d frames -> train %d (%.1f%%) / val %d (%.1f%%)"
          % (SPLIT_SEED, len(shuffled), len(train_ids),
             100.0 * len(train_ids) / len(shuffled),
             len(val_ids), 100.0 * len(val_ids) / len(shuffled)))
    print("wrote", sets / "train.txt", "|", sets / "val.txt", "|", sets / "test.txt")


## 4b. Generate OpenPCDet infos from the raw KITTI data

The archive is a raw KITTI / DAIR-V2X export (velodyne + label_2 + calib + image_2). This cell
turns it into the files the rest of the notebook (CustomDataset) consumes:

* `custom_infos_{train,val}.pkl` - generated with OpenPCDet's `KittiDataset`, which applies the
  calibration to convert the camera-frame labels into LiDAR-frame `gt_boxes_lidar`
* `custom_dbinfos_train.pkl` + `gt_database/*.bin` - the ground-truth database for `gt_sampling`
* `points/<id>.npy` - LiDAR point clouds in the layout `CustomDataset.get_lidar` expects

`KEEP_CLASSES` (section 1) filters the classes; the default is Car only.


In [ ]:
# Convert the raw KITTI-format dataset into the "custom" infos the notebook expects.
if HAVE_RAW and not data_ready(DATA):
    import yaml

    tools = OPCD / "tools"
    raw_cfg = {
        "DATASET": "KittiDataset",
        "DATA_PATH": str(DATA),
        # only used while building infos; the actual training range is derived from
        # your boxes in section 7 - adjust this one if your lidar sits elsewhere
        "POINT_CLOUD_RANGE": [0.0, -39.68, -3.0, 69.12, 39.68, 1.0],
        "POINT_FEATURE_ENCODING": {
            "encoding_type": "absolute_coordinates_encoding",
            "used_feature_list": ["x", "y", "z", "intensity"],
            "src_feature_list": ["x", "y", "z", "intensity"],
        },
        "DATA_SPLIT": {"train": "train", "val": "val", "test": "test"},
        "INFO_PATH": {"train": ["kitti_infos_train.pkl"], "val": ["kitti_infos_val.pkl"],
                      "test": ["kitti_infos_test.pkl"]},
        "DATA_PROCESSOR": [],
        "FOV_POINTS_ONLY": False,
        "GET_ITEM_LIST": ["points"],
    }
    (DATA / "kitti_raw.yaml").write_text(yaml.safe_dump(raw_cfg, sort_keys=False))

    MAKE_INFOS = r"""import os, pickle
from pathlib import Path
import numpy as np, yaml
from easydict import EasyDict
from pcdet.datasets.kitti.kitti_dataset import KittiDataset
import pcdet.datasets.kitti.kitti_dataset as _kd
_kd.Path = Path  # upstream bug: create_groundtruth_database() uses Path without importing it

DATA = Path(os.environ["SHIP_DATA"])
KEEP = [c for c in os.environ.get("SHIP_KEEP_CLASSES", "").split(",") if c]
cfg = EasyDict(yaml.safe_load(open(DATA / "kitti_raw.yaml")))
ds = KittiDataset(dataset_cfg=cfg, class_names=["Car", "Pedestrian", "Cyclist"],
                  root_path=DATA, training=False)


def keep_classes(infos):
    if not KEEP:
        return infos
    kept_total, seen = 0, set()
    for info in infos:
        a = info.get("annos")
        if not a or "name" not in a:
            continue
        mask = np.array([str(n) in KEEP for n in a["name"]], dtype=bool)
        seen.update(str(n) for n in a["name"])
        kept_total += int(mask.sum())
        n = len(a["name"])
        for k in list(a.keys()):
            try:
                if len(a[k]) == n:
                    a[k] = np.asarray(a[k])[mask]
            except TypeError:
                pass
    if kept_total == 0:
        raise RuntimeError(
            "KEEP_CLASSES=%r matched 0 annotations. Classes found in the data: %s. "
            "Edit KEEP_CLASSES in the Config cell (None = keep every class)."
            % (KEEP, sorted(seen)))
    return infos


ds.set_split("train")
infos_train = keep_classes(ds.get_infos(num_workers=2, has_label=True, count_inside_pts=True))
ds.set_split("val")
infos_val = keep_classes(ds.get_infos(num_workers=2, has_label=True, count_inside_pts=True))

pickle.dump(infos_train, open(DATA / "custom_infos_train.pkl", "wb"))
pickle.dump(infos_val, open(DATA / "custom_infos_val.pkl", "wb"))

ds.set_split("train")
ds.create_groundtruth_database(str(DATA / "custom_infos_train.pkl"),
                               used_classes=(KEEP or None), split="train")
os.replace(DATA / "kitti_dbinfos_train.pkl", DATA / "custom_dbinfos_train.pkl")
print("INFOS_DONE")
"""
    (WORK / "make_custom_infos.py").write_text(MAKE_INFOS)

    conv_env = {
        "PCDET_USE_SHIM": "1" if USE_SHIM else "0",
        "PYTHONPATH": "%s:%s:%s:%s" % (tools, OPCD, WORK, os.environ.get("PYTHONPATH", "")),
        "LD_LIBRARY_PATH": "%s:%s" % (str(Path(torch.__file__).parent / "lib"),
                                      os.environ.get("LD_LIBRARY_PATH", "")),
        "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",
        "SHIP_DATA": str(DATA),
        "SHIP_KEEP_CLASSES": ",".join(KEEP_CLASSES or []),
    }
    conv_log = sh("%s %s/run_pcdet.py %s" % (sys.executable, WORK, WORK / "make_custom_infos.py"),
                  cwd=str(tools), env=conv_env)
    assert "INFOS_DONE" in conv_log, "info generation failed - see the log above"

    # velodyne/*.bin -> points/<id>.npy (what CustomDataset.get_lidar reads).
    # The .bin copy is dropped afterwards: it holds identical data and this dataset
    # needs every GB of the Colab disk (~32 GB of bins).
    pts_dir = DATA / "points"
    pts_dir.mkdir(exist_ok=True)
    velo = sorted((DATA / "training" / "velodyne").glob("*.bin"))
    freed = 0
    for f in velo:
        arr = np.fromfile(f, dtype=np.float32).reshape(-1, 4)
        np.save(pts_dir / (f.stem + ".npy"), arr.astype(np.float32))
        if FREE_DISK_AFTER_CONVERT:
            freed += f.stat().st_size
            f.unlink()
    print("wrote %d point files to %s" % (len(velo), pts_dir))
    if freed:
        print("freed %.1f GB by dropping the velodyne .bin copies" % (freed / 1e9))

assert data_ready(DATA), "dataset is incomplete - re-run section 2 then 4b"
print("dataset ready:", DATA)
for p in sorted(DATA.iterdir()):
    print("  ", p.name)


## 5. Inspect the dataset


In [ ]:
assert data_ready(DATA)
infos_tr = pickle.load(open(DATA / "custom_infos_train.pkl", "rb"))
infos_va = pickle.load(open(DATA / "custom_infos_val.pkl", "rb"))
dbinfos = pickle.load(open(DATA / "custom_dbinfos_train.pkl", "rb"))

CLASS_NAMES = [c for c in dbinfos.keys() if c != "DontCare"]
assert CLASS_NAMES, "custom_dbinfos_train.pkl contains no usable classes"
NUM_FEATURES = int(infos_tr[0].get("point_cloud", {}).get("num_features", 4))
print("train frames: %d | val frames: %d" % (len(infos_tr), len(infos_va)))
print("classes:", CLASS_NAMES, "| point features:", NUM_FEATURES)
assert NUM_FEATURES in (3, 4), "this notebook assumes x,y,z[,intensity]"

_db_paths = [d["path"] for lst in dbinfos.values() for d in lst if "path" in d]
assert _db_paths, "custom_dbinfos_train.pkl contains no gt_database entries"
p0 = _db_paths[0]
print("example gt_database path:", p0)
assert not os.path.isabs(p0) and ":" not in p0, (
    "custom_dbinfos_train.pkl holds absolute (Windows) paths - regenerate it or fix the paths"
)

boxes_all, names_all = [], []
for inf in infos_tr:
    a = inf.get("annos")
    if not a or len(a["name"]) == 0:
        continue
    boxes_all.append(np.asarray(a["gt_boxes_lidar"], dtype=np.float64))
    names_all.append(np.asarray(a["name"]))
assert boxes_all, "no annotated training frames found"
boxes_all = np.concatenate(boxes_all, axis=0)
names_all = np.concatenate(names_all, axis=0)

ANCH = {}
for c in CLASS_NAMES:
    b = boxes_all[names_all == c]
    if len(b) == 0:
        continue
    ANCH[c] = {
        "n": int(len(b)),
        "size": [float(v) for v in b[:, 3:6].mean(0).round(2)],
        "bottom": float((b[:, 2] - b[:, 5] / 2).mean().round(2)),
    }
    print("%-14s n=%5d  mean l,w,h=%s  mean bottom z=%s  max l=%.1f"
          % (c, ANCH[c]["n"], ANCH[c]["size"], ANCH[c]["bottom"], b[:, 3].max()))

all_names = sorted(set(str(n) for n in names_all))
print("all annotation names:", all_names)
print("box centre x: %.1f .. %.1f | y: %.1f .. %.1f | z: %.1f .. %.1f"
      % (boxes_all[:, 0].min(), boxes_all[:, 0].max(),
         boxes_all[:, 1].min(), boxes_all[:, 1].max(),
         boxes_all[:, 2].min(), boxes_all[:, 2].max()))

points_dir = DATA / "points"
assert points_dir.is_dir(), "missing %s" % points_dir
pfiles = sorted(p for p in points_dir.iterdir() if p.suffix in (".bin", ".npy"))
assert pfiles, "no point files in %s" % points_dir
PEXT = pfiles[0].suffix
print("%d point files (%s)" % (len(pfiles), PEXT))

## 6. Prepare points (`points/*.npy`, intensity in 0-1), clean `DontCare`, write `ImageSets`

* `CustomDataset.get_lidar` reads `points/<lidar_idx>.npy` -> `.bin` files are converted once.
* Intensity is normalised to 0-1 (KITTI-like) if it clearly is not already.
* `DontCare` annotations are dropped (they have no valid 3D box and would poison the evaluator).
* `ImageSets/{train,val}.txt` are generated **in infos order** (CustomDataset indexes them positionally).


In [ ]:
points_dir = DATA / "points"

def load_pts(f):
    f = Path(f)
    if f.suffix == ".npy":
        return np.load(f).astype(np.float32)
    return np.fromfile(f, dtype=np.float32).reshape(-1, NUM_FEATURES)

pfiles = sorted(p for p in points_dir.iterdir() if p.suffix in (".bin", ".npy"))
PEXT = pfiles[0].suffix

imax = 0.0
if NUM_FEATURES == 4:
    for f in pfiles[:50]:
        try:
            imax = max(imax, float(load_pts(f)[:, 3].max()))
        except Exception:
            pass
scale = None
if NUM_FEATURES == 4 and imax > 1.5:
    scale = 255.0 if imax <= 255 else imax
print("%d point files (%s) | max intensity over first 50 frames = %g | scale = %s"
      % (len(pfiles), PEXT, imax, scale))

marker = DATA / ".prepared_v2"
if not marker.exists():
    if PEXT == ".bin" or scale is not None:
        for f in pfiles:
            a = load_pts(f)
            if scale is not None:
                a[:, 3] = np.clip(a[:, 3] / scale, 0.0, 1.0)
            np.save(f.with_suffix(".npy"), a.astype(np.float32))
    if scale is not None:
        gtdb = DATA / "gt_database"
        if gtdb.is_dir():
            for f in gtdb.glob("*.bin"):
                a = np.fromfile(f, dtype=np.float32).reshape(-1, NUM_FEATURES)
                a[:, 3] = np.clip(a[:, 3] / scale, 0.0, 1.0)
                a.astype(np.float32).tofile(f)
    marker.write_text("done")

npy_files = sorted(points_dir.glob("*.npy"))
assert npy_files, "no .npy point files were produced"


def strip_dontcare(infos):
    removed = 0
    for info in infos:
        a = info.get("annos")
        if not a or "name" not in a:
            continue
        keep = np.array([str(n) != "DontCare" for n in a["name"]], dtype=bool)
        if keep.all():
            continue
        n = len(a["name"])
        removed += int((~keep).sum())
        for k, v in list(a.items()):
            try:
                if len(v) == n:
                    a[k] = np.asarray(v)[keep]
            except TypeError:
                pass
    return removed


for split in ("train", "val"):
    p = DATA / ("custom_infos_%s.pkl" % split)
    infos = pickle.load(open(p, "rb"))
    r = strip_dontcare(infos)
    pickle.dump(infos, open(p, "wb"))
    if r:
        print("%s: removed %d DontCare annotations" % (split, r))

for split in ("train", "val"):
    infos = pickle.load(open(DATA / ("custom_infos_%s.pkl" % split), "rb"))
    ids = [str(i["point_cloud"]["lidar_idx"]) for i in infos]
    missing = [s for s in ids if not (points_dir / (s + ".npy")).exists()]
    assert not missing, (
        "%d point files referenced by custom_infos_%s.pkl are missing (e.g. %s). "
        "Example available file: %s" % (len(missing), split, missing[:5], npy_files[0].name))
    (DATA / "ImageSets").mkdir(exist_ok=True)
    (DATA / "ImageSets" / (split + ".txt")).write_text("\n".join(ids) + "\n")
    print("ImageSets/%s.txt: %d entries" % (split, len(ids)))

print("prepared. npy files:", len(npy_files))

## 7. Write the OpenPCDet configs

Class names, anchors, point-cloud range, voxel size and the KITTI class mapping are derived from the data.


In [ ]:
import yaml

infos_tr = pickle.load(open(DATA / "custom_infos_train.pkl", "rb"))
dbinfos = pickle.load(open(DATA / "custom_dbinfos_train.pkl", "rb"))
CLASS_NAMES = [c for c in dbinfos.keys() if c != "DontCare"]
NUM_FEATURES = int(infos_tr[0].get("point_cloud", {}).get("num_features", 4))

boxes_all, names_all = [], []
for inf in infos_tr:
    a = inf.get("annos")
    if not a or len(a["name"]) == 0:
        continue
    boxes_all.append(np.asarray(a["gt_boxes_lidar"], dtype=np.float64))
    names_all.append(np.asarray(a["name"]))
boxes_all = np.concatenate(boxes_all, axis=0)
names_all = np.concatenate(names_all, axis=0)

ANCH = {}
for c in CLASS_NAMES:
    b = boxes_all[names_all == c]
    if len(b) == 0:
        continue
    ANCH[c] = {
        "size": [float(v) for v in b[:, 3:6].mean(0).round(2)],
        "bottom": float((b[:, 2] - b[:, 5] / 2).mean().round(2)),
    }
CLASS_NAMES = [c for c in CLASS_NAMES if c in ANCH]
assert CLASS_NAMES
if len(CLASS_NAMES) > 6:
    _counts = {c: int((names_all == c).sum()) for c in CLASS_NAMES}
    _ranked = sorted(CLASS_NAMES, key=lambda c: _counts[c], reverse=True)
    print("WARNING: the KITTI evaluator supports 6 classes. Keeping the 6 most frequent and "
          "dropping: %s" % _ranked[6:])
    CLASS_NAMES = _ranked[:6]

# ---- point cloud range / voxel size -----------------------------------------
pad = 10.0
x0, x1 = float(boxes_all[:, 0].min() - pad), float(boxes_all[:, 0].max() + pad)
y0, y1 = float(boxes_all[:, 1].min() - pad), float(boxes_all[:, 1].max() + pad)
z0 = float((boxes_all[:, 2] - boxes_all[:, 5] / 2).min() - 1.0)
z1 = float((boxes_all[:, 2] + boxes_all[:, 5] / 2).max() + 1.0)

side = max(x1 - x0, y1 - y0)
VXY_AUTO = max(0.16, math.ceil((side / 512.0) * 100) / 100.0)
if VOXEL_XY is not None:
    VXY = float(VOXEL_XY)
    print("VOXEL_XY override from the Config cell: %.2f m "
          "(auto for a %.0f m wide scene would be %.2f m)" % (VXY, side, VXY_AUTO))
else:
    VXY = VXY_AUTO
cell = VXY * 16                                   # backbone needs a grid divisible by 16
x0, x1 = math.floor(x0 / cell) * cell, math.ceil(x1 / cell) * cell
y0, y1 = math.floor(y0 / cell) * cell, math.ceil(y1 / cell) * cell
z0, z1 = float(math.floor(z0)), float(math.ceil(z1))
PCR = [round(v, 4) for v in (x0, y0, z0, x1, y1, z1)]
VOXEL = [round(VXY, 4), round(VXY, 4), round(z1 - z0, 4)]
GX, GY = int(round((x1 - x0) / VXY)), int(round((y1 - y0) / VXY))
assert GX % 16 == 0 and GY % 16 == 0, (GX, GY)
print("POINT_CLOUD_RANGE:", PCR)
print("VOXEL_SIZE:", VOXEL, "| BEV grid:", GX, "x", GY)

# ---- KITTI class mapping (the evaluator only understands 6 KITTI names) -----
KITTI_EVAL_NAMES = ["Car", "Pedestrian", "Cyclist", "Van", "Person_sitting", "Truck"]
assert len(CLASS_NAMES) <= len(KITTI_EVAL_NAMES), (
    "the KITTI-style evaluator supports at most 6 classes, dataset has %d" % len(CLASS_NAMES))
MAP_CLASS_TO_KITTI = {c: KITTI_EVAL_NAMES[i] for i, c in enumerate(CLASS_NAMES)}
MAP_CLASS_TO_KITTI["DontCare"] = "DontCare"
print("class -> KITTI:", MAP_CLASS_TO_KITTI)

# ---- gt_sampling groups (guard against classes with too few database points) -
def _count(c, thr):
    return sum(1 for d in dbinfos.get(c, []) if int(d.get("num_points_in_gt", 0)) >= thr)

sample_groups, min_point_filters = [], []
for c in CLASS_NAMES:
    thr = 5 if _count(c, 5) > 0 else 1
    if _count(c, thr) > 0:
        sample_groups.append("%s:10" % c)
        min_point_filters.append("%s:%d" % (c, thr))
    else:
        print("note: class '%s' has no usable gt_database entries -> no gt_sampling for it" % c)

feat = ["x", "y", "z", "intensity"][:NUM_FEATURES]
aug_list = []
if sample_groups:
    aug_list.append({
        "NAME": "gt_sampling", "USE_ROAD_PLANE": False,
        "DB_INFO_PATH": ["custom_dbinfos_train.pkl"],
        "PREPARE": {"filter_by_min_points": min_point_filters},
        "SAMPLE_GROUPS": sample_groups,
        "NUM_POINT_FEATURES": NUM_FEATURES, "DATABASE_WITH_FAKELIDAR": False,
        "REMOVE_EXTRA_WIDTH": [0.0, 0.0, 0.0], "LIMIT_WHOLE_SCENE": True,
    })
aug_list += [
    {"NAME": "random_world_flip", "ALONG_AXIS_LIST": ["x"]},
    {"NAME": "random_world_rotation", "WORLD_ROT_ANGLE": [-0.78539816, 0.78539816]},
    {"NAME": "random_world_scaling", "WORLD_SCALE_RANGE": [0.95, 1.05]},
]

dataset_cfg = {
    "DATASET": "CustomDataset",
    "DATA_PATH": str(DATA),
    "POINT_CLOUD_RANGE": PCR,
    "MAP_CLASS_TO_KITTI": MAP_CLASS_TO_KITTI,
    "DATA_SPLIT": {"train": "train", "test": "val"},
    "INFO_PATH": {"train": ["custom_infos_train.pkl"], "test": ["custom_infos_val.pkl"]},
    "POINT_FEATURE_ENCODING": {"encoding_type": "absolute_coordinates_encoding",
                               "used_feature_list": feat, "src_feature_list": feat},
    "DATA_AUGMENTOR": {
        "DISABLE_AUG_LIST": ["placeholder"],
        "AUG_CONFIG_LIST": aug_list,
    },
    "DATA_PROCESSOR": [
        {"NAME": "mask_points_and_boxes_outside_range", "REMOVE_OUTSIDE_BOXES": True},
        {"NAME": "shuffle_points", "SHUFFLE_ENABLED": {"train": True, "test": False}},
        {"NAME": "transform_points_to_voxels", "VOXEL_SIZE": VOXEL, "MAX_POINTS_PER_VOXEL": 32,
         "MAX_NUMBER_OF_VOXELS": {"train": MAX_VOXELS_TRAIN, "test": MAX_VOXELS_TEST}},
    ],
}

anchor_cfg = [{
    "class_name": c, "anchor_sizes": [ANCH[c]["size"]],
    "anchor_rotations": [0, 1.57], "anchor_bottom_heights": [ANCH[c]["bottom"]],
    "align_center": False, "feature_map_stride": 2,
    "matched_threshold": 0.6, "unmatched_threshold": 0.45,
} for c in CLASS_NAMES]

tools = OPCD / "tools"
(tools / "cfgs" / "custom_models").mkdir(parents=True, exist_ok=True)

model_cfg = {
    "CLASS_NAMES": CLASS_NAMES,
    "DATA_CONFIG": {"_BASE_CONFIG_": str(tools / "cfgs" / "dataset_configs" / "ship_dataset.yaml")},
    "MODEL": {
        "NAME": "PointPillar",
        "VFE": {"NAME": "PillarVFE", "WITH_DISTANCE": False, "USE_ABSLOTE_XYZ": True,
                "USE_NORM": True, "NUM_FILTERS": [64]},
        "MAP_TO_BEV": {"NAME": "PointPillarScatter", "NUM_BEV_FEATURES": 64},
        "BACKBONE_2D": {"NAME": "BaseBEVBackbone", "LAYER_NUMS": [3, 5, 5],
                        "LAYER_STRIDES": [2, 2, 2], "NUM_FILTERS": [64, 128, 256],
                        "UPSAMPLE_STRIDES": [1, 2, 4], "NUM_UPSAMPLE_FILTERS": [128, 128, 128]},
        "DENSE_HEAD": {
            "NAME": "AnchorHeadSingle", "CLASS_AGNOSTIC": False, "USE_DIRECTION_CLASSIFIER": True,
            "DIR_OFFSET": 0.78539, "DIR_LIMIT_OFFSET": 0.0, "NUM_DIR_BINS": 2,
            "ANCHOR_GENERATOR_CONFIG": anchor_cfg,
            "TARGET_ASSIGNER_CONFIG": {"NAME": "AxisAlignedTargetAssigner", "POS_FRACTION": -1.0,
                                       "SAMPLE_SIZE": 512, "NORM_BY_NUM_EXAMPLES": False,
                                       "MATCH_HEIGHT": False, "BOX_CODER": "ResidualCoder"},
            "LOSS_CONFIG": {"LOSS_WEIGHTS": {"cls_weight": 1.0, "loc_weight": 2.0,
                                             "dir_weight": 0.2, "code_weights": [1.0] * 7}},
        },
        "POST_PROCESSING": {"RECALL_THRESH_LIST": [0.3, 0.5, 0.7], "SCORE_THRESH": 0.1,
                            "OUTPUT_RAW_SCORE": False, "EVAL_METRIC": "kitti",
                            "NMS_CONFIG": {"MULTI_CLASSES_NMS": False, "NMS_TYPE": "nms_gpu",
                                           "NMS_THRESH": 0.01, "NMS_PRE_MAXSIZE": 4096,
                                           "NMS_POST_MAXSIZE": 500}},
    },
    "OPTIMIZATION": {"BATCH_SIZE_PER_GPU": BATCH, "NUM_EPOCHS": EPOCHS, "OPTIMIZER": "adam_onecycle",
                     "LR": 0.003, "WEIGHT_DECAY": 0.01, "MOMENTUM": 0.9, "MOMS": [0.95, 0.85],
                     "PCT_START": 0.4, "DIV_FACTOR": 10, "DECAY_STEP_LIST": [35, 45],
                     "LR_DECAY": 0.1, "LR_CLIP": 1e-7, "LR_WARMUP": False, "WARMUP_EPOCH": 1,
                     "GRAD_NORM_CLIP": 10},
}

clean = lambda d: json.loads(json.dumps(d, default=float))
(tools / "cfgs" / "dataset_configs" / "ship_dataset.yaml").write_text(
    yaml.safe_dump(clean(dataset_cfg), sort_keys=False))
(tools / "cfgs" / "custom_models" / "ship_pointpillar.yaml").write_text(
    yaml.safe_dump(clean(model_cfg), sort_keys=False))
CFG_REL = "cfgs/custom_models/ship_pointpillar.yaml"
print("wrote", tools / CFG_REL)

# ---- pre-flight: the anchor grid must exactly tile the BEV feature map ------------------
# Loads anchor_generator.py directly (no pcdet import chain) and checks the invariant that
# the section-3b patch guarantees. On an UNPATCHED file this raises right here - with the
# fix named - instead of killing the first training step with a focal-loss shape crash.
import importlib.util as _ilu
_spec = _ilu.spec_from_file_location(
    "_ag_mod", OPCD / "pcdet" / "models" / "dense_heads" / "target_assigner" / "anchor_generator.py")
_ag_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_ag_mod)
_fm = [[GX // int(a["feature_map_stride"]), GY // int(a["feature_map_stride"])] for a in anchor_cfg]
_ag = _ag_mod.AnchorGenerator(anchor_range=np.array(PCR, dtype=np.float32),
                              anchor_generator_config=anchor_cfg)
_anchors, _n_per_loc = _ag.generate_anchors(_fm)
for _anc, _npl, _fk in zip(_anchors, _n_per_loc, _fm):
    _expect = int(_fk[0]) * int(_fk[1]) * int(_npl)
    _got = int(np.prod(_anc.shape[:-1]))
    assert _got == _expect, (
        "anchor grid produced %d anchors but the BEV feature map needs %d "
        "(%d x %d cells x %d anchors) -> re-run the section 3b patch cell"
        % (_got, _expect, _fk[0], _fk[1], _npl))
print("pre-flight OK: %d anchors == %d x %d feature cells x %d anchors"
      % (int(np.prod(_anchors[0].shape[:-1])), _fm[0][0], _fm[0][1], _n_per_loc[0]))


## 8. Pretrained KITTI PointPillars weights (optional)

Only tensors with matching shapes are loaded; the rest keep their random init.
If the download fails it simply trains from scratch.


In [ ]:
CKPT_PRE = WORK / "pointpillar_7728.pth"
PRE_FLAG = ""
if USE_PRETRAINED:
    if not CKPT_PRE.exists():
        sh('gdown 1wMxWTpU1qUoY3DsCH31WJmvJxcjFXKlm -O "%s"' % CKPT_PRE, check=False, quiet=True)
    if CKPT_PRE.exists() and CKPT_PRE.stat().st_size > 1e6:
        PRE_FLAG = "--pretrained_model %s" % CKPT_PRE
print("pretrained flag:", PRE_FLAG or "(training from scratch)")

## 10. Train


In [ ]:
ENV = {
    "PCDET_USE_SHIM": "1" if USE_SHIM else "0",
    "PYTHONPATH": "%s:%s:%s:%s" % (tools, OPCD, WORK, os.environ.get("PYTHONPATH", "")),
    "LD_LIBRARY_PATH": "%s:%s" % (str(Path(torch.__file__).parent / "lib"),
                                  os.environ.get("LD_LIBRARY_PATH", "")),
    "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",
    # the 1600x1280 BEV grid is very memory hungry - reduce allocator fragmentation
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
}


def _train_cmd(batch):
    # --use_amp (mixed precision): at fp32 the backbone holds ~13 GB of activations for
    # the backward pass (the 800x640x384 concat alone is ~1.6 GB per batch item), which
    # does not fit a 16 GB Colab GPU. fp16 roughly halves it and is faster on TensorCores.
    return ('%s %s/run_pcdet.py train.py --cfg_file %s --batch_size %d --epochs %d --workers %d '
            '--extra_tag %s --ckpt_save_interval %d --max_ckpt_save_num %d --fix_random_seed '
            '--use_amp %s'
            % (sys.executable, WORK, CFG_REL, batch, EPOCHS, WORKERS, EXP_TAG,
               CKPT_SAVE_INTERVAL, MAX_CKPT_SAVE_NUM, PRE_FLAG))


train_cmd = _train_cmd(BATCH)
print("Executing:", train_cmd, flush=True)
try:
    TRAIN_LOG = sh(train_cmd, cwd=str(tools), env=ENV)
except RuntimeError as e:
    if "CUDA out of memory" not in str(e):
        raise
    prev, BATCH = BATCH, max(1, BATCH // 2)
    if BATCH == prev:
        raise
    print("\nCUDA out of memory at batch_size=%d -> retrying once with batch_size=%d"
          % (prev, BATCH), flush=True)
    train_cmd = _train_cmd(BATCH)
    print("Executing:", train_cmd, flush=True)
    TRAIN_LOG = sh(train_cmd, cwd=str(tools), env=ENV)
print("\ntraining finished")


## 11. Evaluate the latest checkpoint on the val split


In [ ]:
OUT = OPCD / "output" / "custom_models" / "ship_pointpillar" / EXP_TAG
ckpts = sorted(glob.glob(str(OUT / "ckpt" / "checkpoint_epoch_*.pth")),
               key=lambda p: int(re.findall(r"checkpoint_epoch_(\d+)\.pth", p)[-1]))
assert ckpts, "no checkpoints under %s - did training finish?" % (OUT / "ckpt")
CKPT = ckpts[-1]
print("evaluating", CKPT)

test_cmd = (
    '%s %s/run_pcdet.py test.py --cfg_file %s --batch_size %d --workers %d --ckpt "%s" --extra_tag %s'
    % (sys.executable, WORK, CFG_REL, BATCH, WORKERS, CKPT, EXP_TAG)
)
TEST_LOG = sh(test_cmd, cwd=str(tools), env=ENV)

## 12. Accuracy summary (IoU >= 0.50 and IoU >= 0.70)

Re-runs OpenPCDet's KITTI evaluator on the saved predictions and prints a clean table
(AP with 40 recall positions at **IoU >= 0.70 and IoU >= 0.50**, moderate difficulty).

In [ ]:
import pandas as pd
import pcdet_shim
pcdet_shim.apply(USE_SHIM)
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets import build_dataloader
from pcdet.utils import common_utils

res_files = sorted(glob.glob(str(OUT / "**" / "result.pkl"), recursive=True), key=os.path.getmtime)
assert res_files, "no result.pkl was produced by the evaluation"
print("reading", res_files[-1])
det_annos = pickle.load(open(res_files[-1], "rb"))

cfg_from_yaml_file(str(tools / CFG_REL), cfg)
eval_logger = common_utils.create_logger()
val_ds, _, _ = build_dataloader(dataset_cfg=cfg.DATA_CONFIG, class_names=cfg.CLASS_NAMES,
                                batch_size=1, dist=False, workers=0,
                                logger=eval_logger, training=False)
print("ground-truth frames: %d | prediction frames: %d" % (len(val_ds), len(det_annos)))

result_str, ap_dict = val_ds.evaluation(det_annos, cfg.CLASS_NAMES, eval_metric="kitti")
print("\n" + result_str)


def parse_kitti_eval(text):
    rows, cur = [], None
    for line in text.splitlines():
        s = line.strip()
        m = re.match(r"^(\S+)\s+(AP_R40|AP)@([\d.]+),\s*([\d.]+),\s*([\d.]+):$", s)
        if m:
            cur = (m.group(1), m.group(2), float(m.group(3)))
            continue
        m2 = re.match(r"^(bbox|bev|3d)\s+AP:(.*)$", s)
        if m2 and cur is not None:
            vals = [float(x) for x in m2.group(2).split(",") if x.strip()]
            if len(vals) == 3:
                rows.append({"class": cur[0], "metric": cur[1], "iou": cur[2],
                             "type": m2.group(1), "easy": vals[0], "moderate": vals[1], "hard": vals[2]})
    return pd.DataFrame(rows)


df = parse_kitti_eval(result_str)
r40 = df[df.metric == "AP_R40"]
assert len(r40), "could not parse the evaluator output"

idx = ["class", "iou"]
tables = {t: r40[r40.type == t].set_index(idx)["moderate"] for t in ("3d", "bev", "bbox")}
tab = pd.concat({"3D AP_R40": tables["3d"], "BEV AP_R40": tables["bev"],
                 "bbox AP_R40": tables["bbox"]}, axis=1)
tab.index.names = ["class", "IoU"]

print("Per-class accuracy (AP with 40 recall positions, moderate difficulty):")
print(tab.round(2).to_string())
print()
summary = tab.groupby(level="IoU").mean().round(2)
summary.index.name = "IoU"
print("Mean accuracy over classes:")
print(summary.to_string())
print()
best = summary["3D AP_R40"]
print("=" * 62)
for iou, v in best.items():
    print("  3D  mAP @ IoU >= %.2f (R40): %6.2f" % (iou, v))
for iou, v in summary["BEV AP_R40"].items():
    print("  BEV mAP @ IoU >= %.2f (R40): %6.2f" % (iou, v))
print("=" * 62)

print("\nRecall (fraction of GT boxes matched, IoU>=0.3 / 0.5 / 0.7):")
for line in TEST_LOG.splitlines():
    if "recall_rcnn_" in line or "recall_roi_" in line:
        print("  " + line.strip())

## 12b. Matched detections at IoU >= 0.50 and IoU >= 0.70 (whole val split)

Reads the saved predictions (`result.pkl`) and greedily matches them (highest score first)
against the ground truth of **every** validation frame - once at **IoU >= 0.50** and once at
**IoU >= 0.70**. Prints how many GT boxes are covered and how many detections are confirmed
at each threshold. Detections below the model's `SCORE_THRESH` are ignored.


In [ ]:
# Whole-val-split matching of detections to ground truth at the two reported thresholds.
import glob, os, pickle
import numpy as np
import torch
import yaml
from pcdet.ops.iou3d_nms.iou3d_nms_utils import boxes_iou3d_gpu

res_files = sorted(glob.glob(str(OUT / "**" / "result.pkl"), recursive=True), key=os.path.getmtime)
assert res_files, "no result.pkl under %s - run the evaluation cell first" % OUT
det_annos = pickle.load(open(res_files[-1], "rb"))
infos_val = pickle.load(open(DATA / "custom_infos_val.pkl", "rb"))
assert len(det_annos) == len(infos_val), (
    "%d predictions vs %d ground-truth frames - re-run the evaluation cell"
    % (len(det_annos), len(infos_val)))

with open(tools / CFG_REL) as f:   # read SCORE_THRESH from the config written in section 7
    SCORE_THR = float(yaml.safe_load(f)["MODEL"]["POST_PROCESSING"].get("SCORE_THRESH", 0.1))
IOU_THRS = (0.50, 0.70)

mismatch = sum(
    1 for k in range(len(infos_val))
    if str(det_annos[k].get("frame_id", "")) not in ("", str(infos_val[k]["point_cloud"]["lidar_idx"]))
)
if mismatch:
    print("WARNING: %d frames where prediction ids do not line up with the ground truth" % mismatch)


def gt_boxes_of(info):
    a = info.get("annos") or {}
    g = np.asarray(a.get("gt_boxes_lidar", np.zeros((0, 7))), dtype=np.float32)
    if g.ndim != 2 or g.shape[1] < 7:
        return np.zeros((0, 7), dtype=np.float32)
    return g[g[:, 3] > 0][:, :7]


def match_counts(iou, scores, thr):
    """greedy 1:1 matching, highest score first -> (matched GT, matched detections)"""
    used = np.zeros(iou.shape[0], dtype=bool)
    n = 0
    for pi in np.argsort(-scores):
        col = np.where(used, -1.0, iou[:, pi])
        gi = int(np.argmax(col))
        if col[gi] >= thr:
            used[gi] = True
            n += 1
        if used.all():
            break
    return n, n


agg = {t: [0, 0, 0, 0] for t in IOU_THRS}   # [gt_total, gt_matched, det_total, det_matched]
n_frames = 0
for info, anno in zip(infos_val, det_annos):
    gt = gt_boxes_of(info)
    pb = np.asarray(anno.get("boxes_lidar", np.zeros((0, 7))), dtype=np.float32)
    ps = np.asarray(anno.get("score", np.zeros((0,))), dtype=np.float32)
    if pb.ndim != 2 or pb.shape[1] < 7:
        pb, ps = np.zeros((0, 7), dtype=np.float32), np.zeros((0,), dtype=np.float32)
    n = min(len(pb), len(ps))
    pb, ps = pb[:n, :7], ps[:n]
    keep = ps >= SCORE_THR
    pb, ps = pb[keep], ps[keep]

    iou = None
    if len(gt) and len(pb):
        iou = boxes_iou3d_gpu(
            torch.as_tensor(gt, dtype=torch.float32, device="cuda"),
            torch.as_tensor(pb, dtype=torch.float32, device="cuda"),
        ).cpu().numpy()

    n_frames += 1
    for t in IOU_THRS:
        mg, mp = (0, 0) if iou is None else match_counts(iou, ps, t)
        agg[t][0] += len(gt)
        agg[t][1] += mg
        agg[t][2] += len(pb)
        agg[t][3] += mp

print()
print("=" * 78)
print("Matched detections over %d val frames (detections with score >= %.2f)" % (n_frames, SCORE_THR))
for t in IOU_THRS:
    g, gm, d, dm = agg[t]
    print("  IoU >= %.2f : GT matched %6d / %6d (%5.1f%%)  |  detections confirmed %6d / %6d (%5.1f%%)"
          % (t, gm, g, 100.0 * gm / max(g, 1), dm, d, 100.0 * dm / max(d, 1)))
print("=" * 78)
print("AP at both thresholds (R40, moderate) is printed in section 12 above.")


## 13. Visualise predictions (BEV) vs ground truth — IoU >= 0.50 / IoU >= 0.70

In [ ]:
import matplotlib.pyplot as plt
import pcdet_shim
pcdet_shim.apply(USE_SHIM)
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets import build_dataloader
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils
from pcdet.ops.iou3d_nms.iou3d_nms_utils import boxes_iou3d_gpu

cfg_from_yaml_file(str(tools / CFG_REL), cfg)
viz_logger = common_utils.create_logger()
ds, _, _ = build_dataloader(dataset_cfg=cfg.DATA_CONFIG, class_names=cfg.CLASS_NAMES, batch_size=1,
                            dist=False, workers=0, logger=viz_logger, training=False)
model = build_network(model_cfg=cfg.MODEL, num_class=len(cfg.CLASS_NAMES), dataset=ds)
model.load_params_from_file(filename=CKPT, logger=viz_logger, to_cpu=False)
model.cuda().eval()


def rect(b):
    cx, cy, _, l, w, _, yaw = b[:7]
    c, s = np.cos(yaw), np.sin(yaw)
    k = np.array([[-l / 2, -w / 2], [-l / 2, w / 2], [l / 2, w / 2], [l / 2, -w / 2], [-l / 2, -w / 2]])
    return k @ np.array([[c, -s], [s, c]]).T + np.array([cx, cy])


def matched_preds(iou, scores, thr):
    """greedy 1:1 matching, highest score first -> set of matched prediction indices"""
    used_gt, out = set(), set()
    for pi in np.argsort(-scores):
        cand = [g for g in range(iou.shape[0]) if g not in used_gt]
        if not cand:
            break
        best = max(cand, key=lambda g: iou[g, pi])
        if iou[best, pi] >= thr:
            used_gt.add(best)
            out.add(int(pi))
    return out


SCORE_THR = 0.3          # detections below this score are not drawn
IOU_THRS = (0.50, 0.70)  # the two thresholds reported in sections 12 / 12b
tot = {t: 0 for t in IOU_THRS}
tot_gt = tot_det = 0

for i in range(min(4, len(ds))):
    batch = ds.collate_batch([ds[i]])
    load_data_to_gpu(batch)
    with torch.no_grad():
        pred, _ = model.forward(batch)
    pts = batch["points"][:, 1:4].cpu().numpy()
    pb = pred[0]["pred_boxes"].cpu().numpy()
    ps = pred[0]["pred_scores"].cpu().numpy()
    gt = batch["gt_boxes"][0].cpu().numpy()
    gt = gt[gt[:, 3] > 0][:, :7]

    keep = ps > SCORE_THR
    pb, ps = pb[keep], ps[keep]

    hits = {t: set() for t in IOU_THRS}
    if len(gt) and len(pb):
        iou = boxes_iou3d_gpu(
            torch.as_tensor(gt, dtype=torch.float32, device="cuda"),
            torch.as_tensor(pb, dtype=torch.float32, device="cuda"),
        ).cpu().numpy()
        hits = {t: matched_preds(iou, ps, t) for t in IOU_THRS}

    n50, n70 = len(hits[0.50]), len(hits[0.70])
    tot[0.50] += n50
    tot[0.70] += n70
    tot_gt += len(gt)
    tot_det += len(pb)

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.scatter(pts[:, 0], pts[:, 1], s=0.3, c="0.6")
    for g in gt:
        r = rect(g)
        ax.plot(r[:, 0], r[:, 1], "g-", lw=2)
    for pi, (b, s) in enumerate(zip(pb, ps)):
        r = rect(b)
        if pi in hits[0.70]:
            ax.plot(r[:, 0], r[:, 1], color="blue", lw=2.0)
        elif pi in hits[0.50]:
            ax.plot(r[:, 0], r[:, 1], color="orange", ls="--", lw=1.5)
        else:
            ax.plot(r[:, 0], r[:, 1], color="red", ls=":", lw=1.2)
    ax.set_aspect("equal")
    ax.set_title("val frame %d: green=GT | blue=IoU>=0.70 | orange=0.50<=IoU<0.70 | red=below 0.50\n"
                 "GT=%d, detections(score>%.1f)=%d, matched IoU>=0.50: %d, IoU>=0.70: %d"
                 % (i, len(gt), SCORE_THR, len(pb), n50, n70))
    print("frame %d: GT=%d | detections(score>%.1f)=%d | matched IoU>=0.50: %d | matched IoU>=0.70: %d"
          % (i, len(gt), SCORE_THR, len(pb), n50, n70))
    plt.show()

print("-" * 78)
print("VISUAL SUMMARY over %d frames: GT=%d | detections=%d | matched IoU>=0.50: %d | matched IoU>=0.70: %d"
      % (min(4, len(ds)), tot_gt, tot_det, tot[0.50], tot[0.70]))


## 14. Keep your trained weights (Colab VMs are wiped when the session ends)


In [ ]:
dst = Path("/content/drive/MyDrive/ship_pointpillar_ckpt")
if Path("/content/drive/MyDrive").exists():
    dst.mkdir(exist_ok=True)
    shutil.copy(CKPT, dst)
    shutil.copy(tools / CFG_REL, dst)
    shutil.copy(tools / "cfgs" / "dataset_configs" / "ship_dataset.yaml", dst)
    print("copied to", dst)
else:
    print("Drive not mounted - download manually:", CKPT)